# FRMCS Forecasting Benchmark



## 1. Experiment Logic

Run cells from top to bottom. The notebook is split into two rounds:

### Round 1: Complete-Data Benchmark
- First 50% of each dataset is the model-development region.
- Inside that 50%, a small validation part is used only to select the best checkpoint.
- Next 20% is the final test region.
- Last 30% is not touched by this notebook because it is reserved for simulation.
- Round 1 uses all 6 active benchmark models: `DSTAGNN`, `STGFormer`, `STG_MFilmFormer`, `GCNMdynamic`, `SBU-LSTM`, `GCRNODE`.
- Final table metrics are averaged over the whole test split, all cells, and the forecast horizons used in this notebook.
- Inference time is measured separately as one forward prediction round, not the whole test set.

### Round 2: Missingness Scenarios
- Round 2 evaluates only the missingness-relevant models: `STG_MFilmFormer`, `GCNMdynamic`, `SBU-LSTM`, `GCRNODE`.
- Inputs are corrupted by scenario-specific missingness generators while targets remain clean.
- Scenarios are `RM`, `TM`, `SM`, and `STM`.
- Scenario sweep CSVs are written separately in `missing_scenario_csvs/`.
- Missing-scenario KPI uses `MAE`, `RMSE`, and `WAPE` over the first 2 forecast steps.

The notebook saves per-run metrics CSVs first. Training-loss plots are regenerated from those CSVs, so you do not need to retrain just to replot. Final test MAE/RMSE/MAPE values stay in `training_kpi_summary.csv`. Missing-data scenario sweeps are written separately as one CSV per scenario in `missing_scenario_csvs/`.



In [1]:
# ============================================================
# 2. Imports And Global Configuration
# ============================================================

from __future__ import annotations

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import copy
import importlib
import math
import random
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

import stg_mfilmformer as _stg_mfilmformer
import gcnm_missing_adapter as _gcnm_missing_adapter

importlib.reload(_stg_mfilmformer)
importlib.reload(_gcnm_missing_adapter)

from stg_mfilmformer import STGMFiLMFormer

from gcnm_missing_adapter import (
    GCRNODEConfig,
    GCRNODEModel,
    GCNMDynamicMulti,
    GCNMMissingConfig,
    MissingPatternConfig,
    SBULSTM,
    STMemoryGraph,
    apply_missing_pattern,
    build_gcnm_loaders,
    build_gcrnode_loaders,
    build_mfilm_loaders,
    build_masked_loaders,
    build_sbu_loaders,
    compute_graph_distance_matrix,
    scale_gcnm_data,
)

# -------------------- Reproducibility --------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------- Dataset Paths --------------------
DATASETS = {
    "STGdata_A": Path("STG_Datasets/STGdata_A/data/STGdata_A.csv"),
    "STGdata_B": Path("STG_Datasets/STGdata_B/data/STGdata_B.csv"),
    "STGdata_C": Path("STG_Datasets/STGdata_C/data/STGdata_C.csv"),
}
CELL_GNB_PATH = Path("STG_Datasets/cell_gnb_relation.csv")

RESULTS_DIR = Path("forecasting_results_missingData")
RUNS_DIR = RESULTS_DIR / "runs"
PLOTS_DIR = RESULTS_DIR / "plots"
MODELS_DIR = RESULTS_DIR / "checkpoints"
for directory in [RESULTS_DIR, RUNS_DIR, PLOTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
for scenario_dir_name in ["RM", "TM", "SM", "STM"]:
    (RUNS_DIR / scenario_dir_name).mkdir(parents=True, exist_ok=True)
    (MODELS_DIR / scenario_dir_name).mkdir(parents=True, exist_ok=True)

TABLE2_CSV = RESULTS_DIR / "training_kpi_summary.csv"
EFFICIENCY_CSV = RESULTS_DIR / "efficiency_mae_summary.csv"
NEW_ERROR_KPI_CSV = RESULTS_DIR / "test_KPI_new_error.csv"
MISSING_SCENARIO_DIR = RESULTS_DIR / "missing_scenario_csvs"
MISSING_SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- Forecasting Setup --------------------
HISTORY = 16
HORIZON = 4
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
HIDDEN_DIM = 64

# Split: first 50% model development, next 20% test, last 30% simulation.
TRAIN_REGION_FRACTION = 0.50
TEST_FRACTION = 0.20
SIMULATION_RESERVED_FRACTION = 0.30

# Good ML practice: select checkpoint using a validation slice inside the first 50%.
# If you want to train on the full first 50% and use the last epoch, set this to 0.0.
VALIDATION_FRACTION_WITHIN_TRAIN_REGION = 0.10

# Default transform flags. Actual preprocessing is selected per model in MODEL_PREPROCESSING below.
# These defaults are only used when no model-specific override is requested.
USE_LOG1P = True
USE_ZSCORE = False
EPS = 1e-6

FEAT_COLS = [
    "density",
    "avg_speed_kmh",
    "CES_count",
    "critical_DL_sum",
    "critical_UL_sum",
    "perform_DL_sum",
    "perform_UL_sum",
]

TARGET_COLS = [
    "critical_DL_sum",
    "critical_UL_sum",
    "perform_DL_sum",
    "perform_UL_sum",
    "CES_count",
]

LOG_FEATURE_COLS = [
    "density",
    "CES_count",
    "critical_DL_sum",
    "critical_UL_sum",
    "perform_DL_sum",
    "perform_UL_sum",
]

MODEL_ORDER = [
    "DSTAGNN",
    "STGFormer",
    "STG_MFilmFormer",
    "GCNMdynamic",
    "SBU-LSTM",
    "GCRNODE",
]

# Per-model preprocessing setup used by training, testing, checkpoint evaluation, and plotting helpers.
MODEL_PREPROCESSING = {
    "DSTAGNN": {"use_log1p": True, "use_zscore": True},
    "STGFormer": {"use_log1p": True, "use_zscore": True},
    "STG_MFilmFormer": {"use_log1p": False, "use_zscore": False},
    "GCNMdynamic": {"use_log1p": False, "use_zscore": False},
    "SBU-LSTM": {"use_log1p": False, "use_zscore": False},
    "GCRNODE": {"use_log1p": False, "use_zscore": False},
}

MISSING_AWARE_MODELS = {"GCNMdynamic", "SBU-LSTM", "GCRNODE"}
GCNM_MODELS = {"GCNMdynamic"}
SBU_MODELS = {"SBU-LSTM"}
GCRNODE_MODELS = {"GCRNODE"}
GCNM_CONFIG = GCNMMissingConfig(
    history=12,
    horizon=HORIZON,
    observed_ratio=0.8,
    missing_ratio=0.2,
    mask_option="RM",
    spatial_k=5,
    nh=3,
    nd=0,
    nw=0,
    tau=12,
    seed=SEED,
)

MFILM_CONFIG = GCNMMissingConfig(
    history=HISTORY,
    horizon=HORIZON,
    observed_ratio=1.0,
    missing_ratio=0.0,
    mask_option="NONE",
    spatial_k=1,
    nh=0,
    nd=0,
    nw=0,
    tau=HISTORY,
    missing_enabled=False,
    seed=SEED,
)

# Missing-pattern control for the missing-data scenarios.
# Enable exactly one scenario at a time.
MISSING_SCENARIOS = {
    "RM": MissingPatternConfig(
        enabled=True,
        pattern="RM",
        missing_ratio=0.2,
        observed_ratio=0.8,
        seed=SEED,
    ),
    "TM": MissingPatternConfig(
        enabled=False,
        pattern="TM",
        missing_ratio=0.2,
        observed_ratio=0.8,
        temporal_block_length=4,
        seed=SEED,
    ),
    "SM": MissingPatternConfig(
        enabled=False,
        pattern="SM",
        missing_ratio=0.2,
        observed_ratio=0.8,
        spatial_node_block=1,
        seed=SEED,
    ),
    "STM": MissingPatternConfig(
        enabled=False,
        pattern="STM",
        missing_ratio=0.2,
        observed_ratio=0.8,
        spatiotemporal_block_length=4,
        spatiotemporal_node_block=1,
        seed=SEED,
    ),
}


def get_active_missing_pattern() -> MissingPatternConfig:
    active = [name for name, cfg in MISSING_SCENARIOS.items() if cfg.enabled]
    if len(active) > 1:
        raise ValueError(f"Enable only one missing scenario at a time, got: {active}")
    if not active:
        return MissingPatternConfig(enabled=False, pattern="NONE", missing_ratio=0.0, observed_ratio=1.0, seed=SEED)
    return MISSING_SCENARIOS[active[0]]


ACTIVE_MISSING_PATTERN = get_active_missing_pattern()

MISSING_EVAL_HORIZON = 4
MISSING_RATIO_GRID = [0.2, 0.4, 0.6, 0.8]
MISSING_SCENARIO_MODELS = ["STG_MFilmFormer", "GCNMdynamic", "SBU-LSTM", "GCRNODE"]
MISSING_SCENARIO_LABELS = {
    "RM": "random_missing_results.csv",
    "TM": "temporal_missing_results.csv",
    "SM": "spatial_missing_results.csv",
    "STM": "spatiotemporal_missing_results.csv",
}


def get_model_preprocessing(model_name: Optional[str] = None) -> Dict[str, bool]:
    if model_name is None:
        return {"use_log1p": USE_LOG1P, "use_zscore": USE_ZSCORE}
    if model_name not in MODEL_PREPROCESSING:
        raise KeyError(f"Missing preprocessing config for model: {model_name}")
    return MODEL_PREPROCESSING[model_name].copy()

print("Datasets:")
for name, path in DATASETS.items():
    print(f"  {name}: {path}")

# ============================================================
# 3. Data Loading, Graph Building, And Temporal Split
# ============================================================

@dataclass
class STGDataBundle:
    dataset_name: str
    X: torch.Tensor
    Y: torch.Tensor
    A_hat: torch.Tensor
    time_values: np.ndarray
    ci_list: List[int]
    x_mean: np.ndarray
    x_std: np.ndarray
    y_mean: np.ndarray
    y_std: np.ndarray
    split: Dict[str, int]
    use_log1p: bool
    use_zscore: bool
    model_name: Optional[str]


def build_adjacency(rel: pd.DataFrame, ci_list: List[int]) -> torch.Tensor:
    """Build a railway-cell adjacency from the anonymized cell-gNB relation."""
    ci_to_idx = {ci: i for i, ci in enumerate(ci_list)}
    n_nodes = len(ci_list)
    adjacency = np.zeros((n_nodes, n_nodes), dtype=np.float32)

    g_to_cells = rel.groupby("gnb")["CI"].apply(lambda s: sorted(s.astype(int).tolist())).to_dict()
    gnb_ids = sorted(g_to_cells.keys())

    def add_undirected(ci_a: int, ci_b: int) -> None:
        if ci_a in ci_to_idx and ci_b in ci_to_idx:
            i, j = ci_to_idx[ci_a], ci_to_idx[ci_b]
            adjacency[i, j] = 1.0
            adjacency[j, i] = 1.0

    # Cells under the same gNB are connected.
    for cells in g_to_cells.values():
        for i in range(len(cells)):
            for j in range(i + 1, len(cells)):
                add_undirected(cells[i], cells[j])

    # Neighbor gNBs are connected along the railway corridor.
    for gnb in gnb_ids:
        next_gnb = gnb + 1
        if next_gnb in g_to_cells:
            for ci_a in g_to_cells[gnb]:
                for ci_b in g_to_cells[next_gnb]:
                    add_undirected(ci_a, ci_b)

    adjacency += np.eye(n_nodes, dtype=np.float32)
    degree = adjacency.sum(axis=1)
    degree_inv_sqrt = np.zeros_like(degree, dtype=np.float32)
    degree_inv_sqrt[degree > 0] = np.power(degree[degree > 0], -0.5)
    normalized = degree_inv_sqrt[:, None] * adjacency * degree_inv_sqrt[None, :]
    return torch.tensor(normalized, dtype=torch.float32, device=DEVICE)


def compute_split_indices(num_steps: int) -> Dict[str, int]:
    train_region_end = int(num_steps * TRAIN_REGION_FRACTION)
    test_end = int(num_steps * (TRAIN_REGION_FRACTION + TEST_FRACTION))

    val_len = int(train_region_end * VALIDATION_FRACTION_WITHIN_TRAIN_REGION)
    if val_len > 0:
        train_core_end = train_region_end - val_len
        val_start = train_core_end
        val_end = train_region_end
    else:
        train_core_end = train_region_end
        val_start = train_region_end
        val_end = train_region_end

    return {
        "train_core_start": HISTORY,
        "train_core_end": train_core_end,
        "val_start": val_start,
        "val_end": val_end,
        "test_start": train_region_end,
        "test_end": test_end,
        "simulation_start": test_end,
        "simulation_end": num_steps,
    }


def load_stg_dataset(
    dataset_name: str,
    model_name: Optional[str] = None,
    use_log1p: Optional[bool] = None,
    use_zscore: Optional[bool] = None,
) -> STGDataBundle:
    dataset_path = DATASETS[dataset_name]
    if not dataset_path.exists():
        raise FileNotFoundError(dataset_path)
    if not CELL_GNB_PATH.exists():
        raise FileNotFoundError(CELL_GNB_PATH)

    df = pd.read_csv(dataset_path)
    rel = pd.read_csv(CELL_GNB_PATH)
    df.columns = [c.strip() for c in df.columns]
    rel.columns = [c.strip() for c in rel.columns]

    missing = sorted(({"time_s", "CI", "gnb"} | set(FEAT_COLS) | set(TARGET_COLS)) - set(df.columns))
    if missing:
        raise ValueError(f"{dataset_name} missing required columns: {missing}")

    rel["CI"] = pd.to_numeric(rel["CI"], errors="coerce").astype("Int64")
    rel["gnb"] = pd.to_numeric(rel["gnb"], errors="coerce").astype("Int64")
    rel = rel.dropna(subset=["CI", "gnb"]).copy()
    rel["CI"] = rel["CI"].astype(int)
    rel["gnb"] = rel["gnb"].astype(int)

    ci_list = sorted(rel["CI"].unique().tolist())
    A_hat = build_adjacency(rel, ci_list)

    df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
    df["CI"] = pd.to_numeric(df["CI"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["time_s", "CI"]).copy()
    df["CI"] = df["CI"].astype(int)
    df = df[df["CI"].isin(ci_list)].sort_values(["time_s", "CI"], kind="stable")

    time_values = np.array(sorted(df["time_s"].unique()))

    def pivot_column(column: str) -> np.ndarray:
        pivot = df.pivot(index="time_s", columns="CI", values=column)
        pivot = pivot.reindex(index=time_values, columns=ci_list)
        return pivot.to_numpy(dtype=np.float32)

    X_raw = np.stack([pivot_column(col) for col in FEAT_COLS], axis=-1)
    Y_raw = np.stack([pivot_column(col) for col in TARGET_COLS], axis=-1)
    X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)
    Y_raw = np.nan_to_num(Y_raw, nan=0.0, posinf=0.0, neginf=0.0)

    preprocess = get_model_preprocessing(model_name)
    if use_log1p is not None:
        preprocess["use_log1p"] = use_log1p
    if use_zscore is not None:
        preprocess["use_zscore"] = use_zscore
    use_log1p = preprocess["use_log1p"]
    use_zscore = preprocess["use_zscore"]

    split = compute_split_indices(len(time_values))
    train_slice = slice(0, max(split["train_core_end"], 1))
    graph_dists = compute_graph_distance_matrix(A_hat.detach().cpu().numpy())

    if model_name in MISSING_AWARE_MODELS:
        X_work, Y_work, x_scale, y_scale = scale_gcnm_data(X_raw, Y_raw, split["train_core_end"])
        use_log1p = False
        use_zscore = False
        x_mean = np.zeros((1, 1, len(FEAT_COLS)), dtype=np.float32)
        x_std = x_scale.astype(np.float32)
        y_mean = np.zeros((1, 1, len(TARGET_COLS)), dtype=np.float32)
        y_std = y_scale.astype(np.float32)
    else:
        X_work = X_raw.copy()
        Y_work = Y_raw.copy()

        if use_log1p:
            log_feature_indices = [FEAT_COLS.index(c) for c in LOG_FEATURE_COLS if c in FEAT_COLS]
            X_work[..., log_feature_indices] = np.log1p(np.maximum(X_work[..., log_feature_indices], 0.0))
            Y_work = np.log1p(np.maximum(Y_work, 0.0))

        if use_zscore:
            x_mean = X_work[train_slice].mean(axis=(0, 1), keepdims=True)
            x_std = X_work[train_slice].std(axis=(0, 1), keepdims=True)
            y_mean = Y_work[train_slice].mean(axis=(0, 1), keepdims=True)
            y_std = Y_work[train_slice].std(axis=(0, 1), keepdims=True)
            x_std = np.where(x_std < EPS, 1.0, x_std)
            y_std = np.where(y_std < EPS, 1.0, y_std)
            X_work = (X_work - x_mean) / x_std
            Y_work = (Y_work - y_mean) / y_std
        else:
            x_mean = np.zeros((1, 1, len(FEAT_COLS)), dtype=np.float32)
            x_std = np.ones((1, 1, len(FEAT_COLS)), dtype=np.float32)
            y_mean = np.zeros((1, 1, len(TARGET_COLS)), dtype=np.float32)
            y_std = np.ones((1, 1, len(TARGET_COLS)), dtype=np.float32)

    print(
        f"{dataset_name}: T={len(time_values):,}, cells={len(ci_list)}, "
        f"train_core=[0:{split['train_core_end']}], "
        f"val=[{split['val_start']}:{split['val_end']}], "
        f"test=[{split['test_start']}:{split['test_end']}], "
        f"simulation=[{split['simulation_start']}:{split['simulation_end']}], "
        f"use_log1p={use_log1p}, use_zscore={use_zscore}, model={model_name}"
    )

    bundle = STGDataBundle(
        dataset_name=dataset_name,
        X=torch.tensor(X_work, dtype=torch.float32),
        Y=torch.tensor(Y_work, dtype=torch.float32),
        A_hat=A_hat,
        time_values=time_values,
        ci_list=ci_list,
        x_mean=x_mean.astype(np.float32),
        x_std=x_std.astype(np.float32),
        y_mean=y_mean.astype(np.float32),
        y_std=y_std.astype(np.float32),
        split=split,
        use_log1p=use_log1p,
        use_zscore=use_zscore,
        model_name=model_name,
    )
    bundle.graph_dists = graph_dists.astype(np.float32)
    return bundle


class WindowDataset(Dataset):
    """Samples one forecasting window with targets fully inside a selected time region."""

    def __init__(self, X: torch.Tensor, Y: torch.Tensor, start: int, end: int):
        self.X = X
        self.Y = Y
        first_target_start = max(int(start), HISTORY)
        last_target_start = int(end) - HORIZON
        if last_target_start < first_target_start:
            self.target_starts = []
        else:
            self.target_starts = list(range(first_target_start, last_target_start + 1))

    def __len__(self) -> int:
        return len(self.target_starts)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        target_start = self.target_starts[index]
        x = self.X[target_start - HISTORY:target_start]
        y = self.Y[target_start:target_start + HORIZON]
        return x, y


def make_loaders(bundle: STGDataBundle) -> Tuple[DataLoader, Optional[DataLoader], DataLoader, DataLoader]:
    split = bundle.split
    if bundle.model_name == "STG_MFilmFormer":
        train_loader, val_loader, test_loader, simulation_loader = build_mfilm_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            MFILM_CONFIG,
        )
        train_size = len(train_loader.dataset)
        val_size = 0 if val_loader is None else len(val_loader.dataset)
        test_size = len(test_loader.dataset)
        simulation_size = len(simulation_loader.dataset)
        print(
            f"STG_MFilmFormer windows: train={train_size:,}, "
            f"val={val_size:,}, "
            f"test={test_size:,}, "
            f"simulation={simulation_size:,}"
        )
        return train_loader, val_loader, test_loader, simulation_loader

    if bundle.model_name in GCNM_MODELS:
        train_loader, val_loader, test_loader, simulation_loader = build_gcnm_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            GCNM_CONFIG,
        )
        train_size = len(train_loader.dataset)
        val_size = 0 if val_loader is None else len(val_loader.dataset)
        test_size = len(test_loader.dataset)
        simulation_size = len(simulation_loader.dataset)
        print(
            f"GCN-M windows: train={train_size:,}, "
            f"val={val_size:,}, "
            f"test={test_size:,}, "
            f"simulation={simulation_size:,}"
        )
        return train_loader, val_loader, test_loader, simulation_loader

    if bundle.model_name in SBU_MODELS:
        train_loader, val_loader, test_loader, simulation_loader = build_sbu_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            SBU_CONFIG,
        )
        train_size = len(train_loader.dataset)
        val_size = 0 if val_loader is None else len(val_loader.dataset)
        test_size = len(test_loader.dataset)
        simulation_size = len(simulation_loader.dataset)
        print(
            f"SBU-LSTM windows: train={train_size:,}, "
            f"val={val_size:,}, "
            f"test={test_size:,}, "
            f"simulation={simulation_size:,}"
        )
        return train_loader, val_loader, test_loader, simulation_loader

    if bundle.model_name in GCRNODE_MODELS:
        train_loader, val_loader, test_loader, simulation_loader = build_gcrnode_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            GCRNODE_CONFIG,
        )
        train_size = len(train_loader.dataset)
        val_size = 0 if val_loader is None else len(val_loader.dataset)
        test_size = len(test_loader.dataset)
        simulation_size = len(simulation_loader.dataset)
        print(
            f"GCRNODE windows: train={train_size:,}, "
            f"val={val_size:,}, "
            f"test={test_size:,}, "
            f"simulation={simulation_size:,}"
        )
        return train_loader, val_loader, test_loader, simulation_loader

    train_ds = WindowDataset(bundle.X, bundle.Y, split["train_core_start"], split["train_core_end"])

    if split["val_end"] > split["val_start"]:
        val_ds = WindowDataset(bundle.X, bundle.Y, split["val_start"], split["val_end"])
    else:
        val_ds = None

    test_ds = WindowDataset(bundle.X, bundle.Y, split["test_start"], split["test_end"])
    simulation_ds = WindowDataset(bundle.X, bundle.Y, split["simulation_start"], split["simulation_end"])

    if len(train_ds) == 0 or len(test_ds) == 0 or len(simulation_ds) == 0:
        raise ValueError("Not enough time steps for HISTORY/HORIZON with the configured split.")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = None if val_ds is None or len(val_ds) == 0 else DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
    simulation_loader = DataLoader(simulation_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

    print(
        f"Windows: train={len(train_ds):,}, "
        f"val={0 if val_loader is None else len(val_loader.dataset):,}, "
        f"test={len(test_ds):,}, "
        f"simulation={len(simulation_ds):,}"
    )
    return train_loader, val_loader, test_loader, simulation_loader

SBU_CONFIG = GCNMMissingConfig(
    history=HISTORY,
    horizon=HORIZON,
    observed_ratio=0.8,
    missing_ratio=0.2,
    mask_option="RM",
    spatial_k=5,
    nh=0,
    nd=0,
    nw=0,
    tau=HISTORY,
    seed=SEED,
)

GCRNODE_CONFIG = GCRNODEConfig(
    history=HISTORY,
    horizon=HORIZON,
    observed_ratio=0.8,
    missing_ratio=0.2,
    mask_option="RM",
    replay_day_size=HISTORY,
    replay_week_size=max(HISTORY * 4, HORIZON * 4),
    memory_dim=32,
    diffusion_depth=2,
    topk=8,
    ode_steps=1,
    ode_solver="euler",
    kl_weight=0.1,
    seed=SEED,
)

apply_missing_pattern(GCNM_CONFIG, ACTIVE_MISSING_PATTERN)
apply_missing_pattern(SBU_CONFIG, ACTIVE_MISSING_PATTERN)
apply_missing_pattern(GCRNODE_CONFIG, ACTIVE_MISSING_PATTERN)

print(
    f"Active missing scenario: pattern={ACTIVE_MISSING_PATTERN.pattern}, "
    f"enabled={ACTIVE_MISSING_PATTERN.enabled}, "
    f"missing_ratio={ACTIVE_MISSING_PATTERN.missing_ratio}"
)



/home/me/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Using device: cpu
Datasets:
  STGdata_A: STG_Datasets/STGdata_A/data/STGdata_A.csv
  STGdata_B: STG_Datasets/STGdata_B/data/STGdata_B.csv
  STGdata_C: STG_Datasets/STGdata_C/data/STGdata_C.csv
Active missing scenario: pattern=RM, enabled=True, missing_ratio=0.2


### Metrics functions


In [2]:
from __future__ import annotations

# ============================================================
# 4. Metrics And Result CSV Helpers
# ============================================================

@dataclass
class MetricResult:
    loss: float
    horizon_mae: np.ndarray
    horizon_rmse: np.ndarray
    horizon_mape: np.ndarray
    average_mae: float
    average_rmse: float
    average_mape: float


def inverse_target_transform(y: torch.Tensor, bundle: STGDataBundle) -> torch.Tensor:
    """Return targets in original traffic units before metric calculation."""
    y_mean = torch.tensor(bundle.y_mean, dtype=torch.float32, device=y.device)
    y_std = torch.tensor(bundle.y_std, dtype=torch.float32, device=y.device)

    if bundle.model_name in MISSING_AWARE_MODELS:
        return y * y_std
    if bundle.use_zscore:
        y = y * y_std + y_mean
    if bundle.use_log1p:
        y = torch.clamp(torch.expm1(y), min=0.0)
    return y


def masked_mape_np(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> float:
    """Traffic forecasting MAPE: ignore zero/near-zero ground-truth entries."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = np.abs(y_true) > eps
    if not np.any(mask):
        return float("nan")
    return float(np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask])) * 100.0)


def metric_triplet_np(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float, float]:
    """Use sklearn for MAE/RMSE and a masked MAPE wrapper for traffic zeros."""
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    mae = float(mean_absolute_error(yt, yp))
    rmse = float(mean_squared_error(yt, yp, squared=False))
    mape = masked_mape_np(yt, yp)
    return mae, rmse, mape


def unpack_model_batch(batch, bundle: STGDataBundle):
    if bundle.model_name in GCNM_MODELS:
        x_stat, x_hist, y = batch
        return x_stat.to(DEVICE), x_hist.to(DEVICE), y.to(DEVICE)
    if bundle.model_name in SBU_MODELS:
        x_masked, mask, x_true, y = batch
        return (x_masked.to(DEVICE), mask.to(DEVICE), x_true.to(DEVICE)), None, y.to(DEVICE)
    if bundle.model_name in GCRNODE_MODELS:
        x_masked, mask, y, hist_idx, fut_idx = batch
        return (x_masked.to(DEVICE), mask.to(DEVICE), hist_idx.to(DEVICE), fut_idx.to(DEVICE)), None, y.to(DEVICE)
    x, y = batch
    return x.to(DEVICE), None, y.to(DEVICE)


def _reshape_sbu_inputs(x: torch.Tensor) -> torch.Tensor:
    b, t, n, f = x.shape
    return x.reshape(b, t, n * f)


def _reshape_sbu_targets(y: torch.Tensor) -> torch.Tensor:
    b, h, n, f = y.shape
    return y.reshape(b, h, n * f)


def _unshape_sbu_preds(pred: torch.Tensor, bundle: STGDataBundle) -> torch.Tensor:
    b, h, flat = pred.shape
    n = len(bundle.ci_list)
    f = len(TARGET_COLS)
    return pred.view(b, h, n, f)


def model_aux_loss(model: nn.Module, bundle: STGDataBundle) -> torch.Tensor:
    if bundle.model_name == "STG_MFilmFormer" and getattr(model, "last_reg_loss", None) is not None:
        return model.recon_lambda * model.last_reg_loss
    if bundle.model_name in SBU_MODELS and getattr(model, "last_reg_loss", None) is not None:
        return model.reg_lambda * model.last_reg_loss
    if bundle.model_name in GCRNODE_MODELS and getattr(model, "last_aux_loss", None) is not None:
        return model.kl_weight * model.last_aux_loss
    return torch.tensor(0.0, device=DEVICE)


def forward_model_batch(model: nn.Module, batch, bundle: STGDataBundle) -> Tuple[torch.Tensor, torch.Tensor]:
    x, x_hist, y = unpack_model_batch(batch, bundle)
    if bundle.model_name in GCNM_MODELS:
        pred = model(x, x_hist)
    elif bundle.model_name in SBU_MODELS:
        x_masked, mask, x_true = x
        pred_flat = model(_reshape_sbu_inputs(x_masked), _reshape_sbu_inputs(mask), _reshape_sbu_inputs(x_true))
        pred = _unshape_sbu_preds(pred_flat, bundle)
    elif bundle.model_name in GCRNODE_MODELS:
        x_masked, mask, hist_idx, fut_idx = x
        pred = model(x_masked, mask, hist_idx, fut_idx)
    elif bundle.model_name == "STG_MFilmFormer":
        pred = model(x, bundle.A_hat)
    else:
        pred = model(x, bundle.A_hat)
    return pred, y


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, bundle: STGDataBundle, loss_fn: nn.Module) -> MetricResult:
    model.eval()
    total_loss = 0.0
    total_samples = 0
    true_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]
    pred_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]

    for batch in loader:
        pred, y = forward_model_batch(model, batch, bundle)
        loss = loss_fn(pred, y) + model_aux_loss(model, bundle)

        batch_size = y.shape[0]
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size

        y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
        y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()

        for h in range(HORIZON):
            true_by_horizon[h].append(y_true[:, h])
            pred_by_horizon[h].append(y_pred[:, h])

    horizon_mae = np.zeros(HORIZON, dtype=np.float64)
    horizon_rmse = np.zeros(HORIZON, dtype=np.float64)
    horizon_mape = np.zeros(HORIZON, dtype=np.float64)

    for h in range(HORIZON):
        yt = np.concatenate(true_by_horizon[h], axis=0)
        yp = np.concatenate(pred_by_horizon[h], axis=0)
        horizon_mae[h], horizon_rmse[h], horizon_mape[h] = metric_triplet_np(yt, yp)

    return MetricResult(
        loss=total_loss / max(total_samples, 1),
        horizon_mae=horizon_mae,
        horizon_rmse=horizon_rmse,
        horizon_mape=horizon_mape,
        average_mae=float(np.nanmean(horizon_mae)),
        average_rmse=float(np.nanmean(horizon_rmse)),
        average_mape=float(np.nanmean(horizon_mape)),
    )


NEW_ERROR_TARGET_GROUPS = {
    "traffic_only": [
        "critical_DL_sum",
        "critical_UL_sum",
        "perform_DL_sum",
        "perform_UL_sum",
    ],
    "emergency_only": [
        "CES_count",
    ],
}
NEW_ERROR_BALANCED_GROUP = "all_targets_balanced"
NEW_ERROR_METRICS = ["mae", "rmse", "mape", "wape"]


def masked_mape_group_np(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> float:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    mask = np.abs(yt) > eps
    if not np.any(mask):
        return float("nan")
    return float(np.mean(np.abs((yp[mask] - yt[mask]) / yt[mask])) * 100.0)


def wape_group_np(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> float:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    denom = np.sum(np.abs(yt))
    if denom <= eps:
        return float("nan")
    return float(np.sum(np.abs(yp - yt)) / denom * 100.0)


def metric_five_np(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    return {
        "mae": float(mean_absolute_error(yt, yp)),
        "rmse": float(mean_squared_error(yt, yp, squared=False)),
        "mape": masked_mape_group_np(yt, yp),
        "wape": wape_group_np(yt, yp),
    }


def empty_new_error_row(dataset_name: str, model_name: str, target_group: str) -> Dict[str, object]:
    row = {"dataset": dataset_name, "model": model_name, "target_group": target_group}
    for h in range(1, HORIZON + 1):
        for metric_name in NEW_ERROR_METRICS:
            row[f"horizon_{h}_{metric_name}"] = np.nan
    for metric_name in NEW_ERROR_METRICS:
        row[f"average_{metric_name}"] = np.nan
    return row


def ensure_new_error_table() -> None:
    if NEW_ERROR_KPI_CSV.exists():
        return
    rows = []
    for dataset_name in DATASETS:
        for model_name in MODEL_ORDER:
            for group_name in ["traffic_only", "emergency_only", NEW_ERROR_BALANCED_GROUP]:
                rows.append(empty_new_error_row(dataset_name, model_name, group_name))
    pd.DataFrame(rows).to_csv(NEW_ERROR_KPI_CSV, index=False)


@torch.no_grad()
def evaluate_model_grouped(model: nn.Module, loaders: List[DataLoader], bundle: STGDataBundle) -> List[Dict[str, object]]:
    model.eval()
    true_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]
    pred_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]

    for loader in loaders:
        for batch in loader:
            pred, y = forward_model_batch(model, batch, bundle)
            y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
            y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()
            for h in range(HORIZON):
                true_by_horizon[h].append(y_true[:, h])
                pred_by_horizon[h].append(y_pred[:, h])

    per_group_per_horizon: Dict[str, Dict[str, List[float]]] = {}
    for group_name in list(NEW_ERROR_TARGET_GROUPS.keys()) + [NEW_ERROR_BALANCED_GROUP]:
        per_group_per_horizon[group_name] = {m: [] for m in NEW_ERROR_METRICS}

    for h in range(HORIZON):
        yt_all = np.concatenate(true_by_horizon[h], axis=0)
        yp_all = np.concatenate(pred_by_horizon[h], axis=0)

        group_metrics: Dict[str, Dict[str, float]] = {}
        for group_name, group_cols in NEW_ERROR_TARGET_GROUPS.items():
            idxs = [TARGET_COLS.index(col) for col in group_cols]
            group_metrics[group_name] = metric_five_np(yt_all[..., idxs], yp_all[..., idxs])
            for metric_name, metric_value in group_metrics[group_name].items():
                per_group_per_horizon[group_name][metric_name].append(metric_value)

        for metric_name in NEW_ERROR_METRICS:
            balanced_value = float(np.nanmean([
                group_metrics["traffic_only"][metric_name],
                group_metrics["emergency_only"][metric_name],
            ]))
            per_group_per_horizon[NEW_ERROR_BALANCED_GROUP][metric_name].append(balanced_value)

    rows: List[Dict[str, object]] = []
    for group_name, metric_dict in per_group_per_horizon.items():
        row = {"target_group": group_name}
        for h in range(1, HORIZON + 1):
            for metric_name in NEW_ERROR_METRICS:
                row[f"horizon_{h}_{metric_name}"] = metric_dict[metric_name][h - 1]
        for metric_name in NEW_ERROR_METRICS:
            row[f"average_{metric_name}"] = float(np.nanmean(metric_dict[metric_name]))
        rows.append(row)
    return rows


def upsert_new_error_rows(dataset_name: str, model_name: str, grouped_rows: List[Dict[str, object]]) -> None:
    ensure_new_error_table()
    df = pd.read_csv(NEW_ERROR_KPI_CSV)
    for grouped_row in grouped_rows:
        group_name = grouped_row["target_group"]
        mask = (df["dataset"] == dataset_name) & (df["model"] == model_name) & (df["target_group"] == group_name)
        if not mask.any():
            df = pd.concat([df, pd.DataFrame([empty_new_error_row(dataset_name, model_name, group_name)])], ignore_index=True)
            mask = (df["dataset"] == dataset_name) & (df["model"] == model_name) & (df["target_group"] == group_name)
        idx = df.index[mask][0]
        for key, value in grouped_row.items():
            df.loc[idx, key] = value
    preferred = ["traffic_only", "emergency_only", NEW_ERROR_BALANCED_GROUP]
    df["target_group"] = pd.Categorical(df["target_group"], categories=preferred, ordered=True)
    df = df.sort_values(["dataset", "model", "target_group"], kind="stable").reset_index(drop=True)
    ordered_cols = ["dataset", "model", "target_group"] + [
        c for c in df.columns if c not in {"dataset", "model", "target_group"}
    ]
    df = df[ordered_cols]
    df.to_csv(NEW_ERROR_KPI_CSV, index=False)


def empty_table2() -> pd.DataFrame:
    rows = []
    for dataset in DATASETS:
        for model in MODEL_ORDER:
            row = {"dataset": dataset, "model": model}
            for h in range(1, HORIZON + 1):
                row[f"horizon_{h}_mae"] = np.nan
                row[f"horizon_{h}_rmse"] = np.nan
                row[f"horizon_{h}_mape"] = np.nan
            row["average_mae"] = np.nan
            row["average_rmse"] = np.nan
            row["average_mape"] = np.nan
            rows.append(row)
    return pd.DataFrame(rows)


def empty_efficiency_table() -> pd.DataFrame:
    rows = []
    for dataset in DATASETS:
        for model in MODEL_ORDER:
            rows.append({
                "dataset": dataset,
                "model": model,
                "flops": np.nan,
                "parameters": np.nan,
                "inference_ms_per_prediction_round": np.nan,
                "avg_mae": np.nan,
            })
    return pd.DataFrame(rows)


def normalize_efficiency_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep backward compatibility with older efficiency CSV column names."""
    if "flops_per_prediction_round" in df.columns and "flops" not in df.columns:
        df = df.rename(columns={"flops_per_prediction_round": "flops"})
    return df


def ensure_efficiency_table() -> None:
    if not EFFICIENCY_CSV.exists():
        empty_efficiency_table().to_csv(EFFICIENCY_CSV, index=False)


def ensure_summary_tables() -> None:
    ensure_efficiency_table()


def upsert_efficiency_row(
    dataset_name: str,
    model_name: str,
    flops: float,
    params: int,
    inference_ms: float,
    avg_mae: float,
) -> None:
    ensure_efficiency_table()
    df = normalize_efficiency_columns(pd.read_csv(EFFICIENCY_CSV))
    mask = (df["dataset"] == dataset_name) & (df["model"] == model_name)
    if not mask.any():
        df = pd.concat([df, pd.DataFrame([{
            "dataset": dataset_name,
            "model": model_name,
            "flops": np.nan,
            "parameters": np.nan,
            "inference_ms_per_prediction_round": np.nan,
            "avg_mae": np.nan,
        }])], ignore_index=True)
        mask = (df["dataset"] == dataset_name) & (df["model"] == model_name)

    idx = df.index[mask][0]
    df.loc[idx, "flops"] = flops
    df.loc[idx, "parameters"] = params
    df.loc[idx, "inference_ms_per_prediction_round"] = inference_ms
    df.loc[idx, "avg_mae"] = avg_mae
    df.to_csv(EFFICIENCY_CSV, index=False)


ensure_efficiency_table()
print(f"Efficiency table: {EFFICIENCY_CSV}")
print(f"Grouped error table: {NEW_ERROR_KPI_CSV}")







Efficiency table: forecasting_results_missingData/efficiency_mae_summary.csv
Grouped error table: forecasting_results_missingData/test_KPI_new_error.csv


### models defintions


In [3]:
from __future__ import annotations

# ============================================================
# 5. Forecasting Models
# ============================================================

# IMPORTANT:
# The first group below restores your original working implementations from
# forecasting_before_stg_benchmark_cleanup.ipynb / train_stgcn_stgformer_stgfilmformer.py.
# Only the surrounding benchmark workflow was changed.

IDX_CES = FEAT_COLS.index("CES_count")
IDX_TRAFFIC = tuple(FEAT_COLS.index(col) for col in [
    "critical_DL_sum",
    "critical_UL_sum",
    "perform_DL_sum",
    "perform_UL_sum",
])

class NodeWiseLSTM(nn.Module):
    """Standard LSTM baseline from the paper family, applied independently per cell."""

    def __init__(self, in_dim: int, out_dim: int, horizon: int, hidden_dim: int = HIDDEN_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.lstm = nn.LSTM(in_dim, hidden_dim, num_layers=2, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon * out_dim),
        )

    def forward(self, x: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        batch_size, history, num_nodes, in_dim = x.shape
        x_seq = x.permute(0, 2, 1, 3).reshape(batch_size * num_nodes, history, in_dim)
        _, (hidden, _) = self.lstm(x_seq)
        out = self.head(hidden[-1])
        out = out.reshape(batch_size, num_nodes, self.horizon, self.out_dim)
        return out.permute(0, 2, 1, 3).contiguous()

# ---------------- STGCN ----------------
class STGCNLayerNorm(nn.Module):
    def __init__(self, num_nodes: int, channels: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(1, 1, num_nodes, channels))
        self.beta = nn.Parameter(torch.zeros(1, 1, num_nodes, channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mu = x.mean(dim=(2, 3), keepdim=True)
        sigma = ((x - mu) ** 2).mean(dim=(2, 3), keepdim=True)
        return (x - mu) / torch.sqrt(sigma + self.eps) * self.gamma + self.beta


class STGCNTemporalConvLayer(nn.Module):
    def __init__(self, Kt: int, c_in: int, c_out: int, act_func: str = 'GLU'):
        super().__init__()
        self.Kt = Kt
        self.c_in = c_in
        self.c_out = c_out
        self.act_func = act_func
        self.res_conv = nn.Conv2d(c_in, c_out, kernel_size=(1, 1)) if c_in > c_out else None
        out_channels = 2 * c_out if act_func == 'GLU' else c_out
        self.conv = nn.Conv2d(c_in, out_channels, kernel_size=(Kt, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, N, _ = x.shape
        x_input = x
        if self.c_in > self.c_out:
            x_input = self.res_conv(x.permute(0, 3, 1, 2).contiguous()).permute(0, 2, 3, 1).contiguous()
        elif self.c_in < self.c_out:
            pad = torch.zeros(B, T, N, self.c_out - self.c_in, device=x.device, dtype=x.dtype)
            x_input = torch.cat([x, pad], dim=-1)
        x_input = x_input[:, self.Kt - 1:T, :, :]
        x_conv = self.conv(x.permute(0, 3, 1, 2).contiguous()).permute(0, 2, 3, 1).contiguous()
        if self.act_func == 'GLU':
            lhs = x_conv[..., :self.c_out]
            rhs = x_conv[..., -self.c_out:]
            return (lhs + x_input) * torch.sigmoid(rhs)
        if self.act_func == 'linear':
            return x_conv
        if self.act_func == 'sigmoid':
            return torch.sigmoid(x_conv)
        if self.act_func == 'relu':
            return torch.relu(x_conv + x_input)
        raise ValueError(f"Unsupported STGCN activation: {self.act_func}")


class STGCNSpatioConvLayer(nn.Module):
    def __init__(self, Ks: int, c_in: int, c_out: int):
        super().__init__()
        self.Ks = Ks
        self.c_in = c_in
        self.c_out = c_out
        self.res_conv = nn.Conv2d(c_in, c_out, kernel_size=(1, 1)) if c_in > c_out else None
        self.theta = nn.Parameter(torch.empty(Ks * c_in, c_out))
        nn.init.xavier_uniform_(self.theta)

    def forward(self, x: torch.Tensor, cheb_polynomials: List[torch.Tensor]) -> torch.Tensor:
        B, T, N, _ = x.shape
        x_input = x
        if self.c_in > self.c_out:
            x_input = self.res_conv(x.permute(0, 3, 1, 2).contiguous()).permute(0, 2, 3, 1).contiguous()
        elif self.c_in < self.c_out:
            pad = torch.zeros(B, T, N, self.c_out - self.c_in, device=x.device, dtype=x.dtype)
            x_input = torch.cat([x, pad], dim=-1)
        outputs = []
        for t in range(T):
            graph_signal = x[:, t]
            cheb_outs = []
            for k in range(self.Ks):
                cheb_outs.append(torch.matmul(cheb_polynomials[k], graph_signal))
            x_ker = torch.cat(cheb_outs, dim=-1)
            outputs.append(torch.matmul(x_ker, self.theta).unsqueeze(1))
        x_gc = torch.cat(outputs, dim=1)
        return torch.relu(x_gc + x_input)


class STGCNBlock(nn.Module):
    def __init__(self, Ks: int, Kt: int, channels: Tuple[int, int, int], num_nodes: int, dropout: float):
        super().__init__()
        c_si, c_t, c_oo = channels
        self.temp_in = STGCNTemporalConvLayer(Kt, c_si, c_t, act_func='GLU')
        self.spat = STGCNSpatioConvLayer(Ks, c_t, c_t)
        self.temp_out = STGCNTemporalConvLayer(Kt, c_t, c_oo, act_func='relu')
        self.norm = STGCNLayerNorm(num_nodes, c_oo)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cheb_polynomials: List[torch.Tensor]) -> torch.Tensor:
        x = self.temp_in(x)
        x = self.spat(x, cheb_polynomials)
        x = self.temp_out(x)
        x = self.norm(x)
        return self.drop(x)


class STGCNOutputLayer(nn.Module):
    def __init__(self, Ko: int, channel: int, num_nodes: int, horizon: int, out_dim: int):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.temp1 = STGCNTemporalConvLayer(Ko, channel, channel, act_func='GLU')
        self.norm = STGCNLayerNorm(num_nodes, channel)
        self.temp2 = STGCNTemporalConvLayer(1, channel, channel, act_func='sigmoid')
        self.fc = nn.Conv2d(channel, horizon * out_dim, kernel_size=(1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.temp1(x)
        x = self.norm(x)
        x = self.temp2(x)
        x = self.fc(x.permute(0, 3, 1, 2).contiguous())
        return x


class STGCN_Yu2018(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, horizon: int, history: int, num_nodes: int, c_t: int = 64, c_mid: int = 32, Ks: int = 3, Kt: int = 3, dropout: float = 0.1):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.history = history
        self.Ks = Ks
        self.in_proj = nn.Linear(in_dim, c_t)
        self.block1 = STGCNBlock(Ks, Kt, (c_t, c_mid, c_t), num_nodes, dropout)
        self.block2 = STGCNBlock(Ks, Kt, (c_t, c_mid, c_t), num_nodes, dropout)
        Ko = history - 2 * 2 * (Kt - 1)
        if Ko <= 1:
            raise ValueError(f"HISTORY={history} too small for STGCN with Kt={Kt}")
        self.output = STGCNOutputLayer(Ko, c_t, num_nodes, horizon, out_dim)
        self._cached_cheb = None

    def forward(self, x_hist: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        B, T, N, _ = x_hist.shape
        if self._cached_cheb is None or self._cached_cheb[0].device != x_hist.device:
            adj = astgcn_support_to_adj(A_hat.to(x_hist.device))
            l_tilde = scaled_laplacian_torch(adj)
            self._cached_cheb = cheb_polynomial_torch(l_tilde, self.Ks)
        x = self.in_proj(x_hist)
        x = self.block1(x, self._cached_cheb)
        x = self.block2(x, self._cached_cheb)
        x = self.output(x).squeeze(2)
        x = x.permute(0, 2, 1).contiguous().view(B, N, self.horizon, self.out_dim)
        return x.permute(0, 2, 1, 3).contiguous()

# ---------------- STGformer ----------------
class LinearSTAttention(nn.Module):
    def __init__(self, dim, dropout=0.0):
        super().__init__()
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mode: str):
        B, T, N, C = x.shape
        if mode == "spatial":
            xt = x.reshape(B * T, N, C)
            Q = self.wq(xt); K = self.wk(xt); V = self.wv(xt)
            Q = Q / (N ** 0.5); K = K / (N ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, T, N, C)
            return self.drop(out)
        elif mode == "temporal":
            xn = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, T, C)
            Q = self.wq(xn); K = self.wk(xn); V = self.wv(xn)
            Q = Q / (T ** 0.5); K = K / (T ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, N, T, C).permute(0, 2, 1, 3).contiguous()
            return self.drop(out)
        else:
            raise ValueError("mode must be 'spatial' or 'temporal'")

class STGAttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.att = LinearSTAttention(dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dim, dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        a = self.att(x, "spatial") + self.att(x, "temporal")
        x = self.norm1(x + a)
        x = self.norm2(x + self.ff(x))
        return x

class STGformer(nn.Module):
    def __init__(self, in_dim, out_dim, horizon, dim=64, K=3, dropout=0.1):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.dim = dim
        self.K = K
        self.in_proj = nn.Linear(in_dim, dim)
        self.time_embed = nn.Parameter(torch.zeros(1, HISTORY, 1, dim))
        nn.init.normal_(self.time_embed, std=0.02)
        self.block = STGAttentionBlock(dim, dropout=dropout)
        self.gate = nn.Linear(dim, dim)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, horizon * out_dim))

    def graph_propagate(self, x, A_hat):
        outs = [x]
        xk = x
        for _ in range(self.K):
            xk = torch.matmul(A_hat, xk)
            outs.append(xk)
        return outs

    def forward(self, x_hist, A_hat):
        B, T, N, F = x_hist.shape
        x = self.in_proj(x_hist) + self.time_embed[:, :T]
        props = self.graph_propagate(x, A_hat)
        h = self.block(props[0])
        for k in range(1, len(props)):
            hk = self.block(props[k])
            g = torch.sigmoid(self.gate(h))
            h = h + g * hk
        h_last = h[:, -1]
        y = self.head(h_last).view(B, N, self.horizon, self.out_dim).permute(0, 2, 1, 3).contiguous()
        return y

# ---------------- STGFiLMFormer ----------------
class EventFiLM(nn.Module):
    """FiLM generator with reusable gamma/beta for shared Q/K/V conditioning."""
    def __init__(self, event_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.mlp = nn.Sequential(
            nn.Linear(event_dim, 2 * hidden_dim),
            nn.ReLU(),
            nn.Linear(2 * hidden_dim, 2 * hidden_dim),
        )

    def gamma_beta(self, e: torch.Tensor):
        gb = self.mlp(e)
        gamma, beta = gb.chunk(2, dim=-1)
        return gamma, beta

    @staticmethod
    def apply_film(x: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor):
        while gamma.dim() < x.dim():
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return gamma * x + beta

    def forward(self, x: torch.Tensor, e: torch.Tensor):
        gamma, beta = self.gamma_beta(e)
        return self.apply_film(x, gamma, beta)

class FiLMLinearSTAttention(nn.Module):
    def __init__(self, dim, event_dim, dropout=0.0):
        super().__init__()
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.drop = nn.Dropout(dropout)
        self.film_qkv = EventFiLM(event_dim, dim)

    def forward(self, x, e, mode: str):
        B, T, N, C = x.shape
        if mode == "spatial":
            xt = x.reshape(B * T, N, C)
            e_bt = e.unsqueeze(1).repeat(1, T, 1, 1).reshape(B * T, N, -1)
            Q0 = self.wq(xt)
            K0 = self.wk(xt)
            V0 = self.wv(xt)
            gamma, beta = self.film_qkv.gamma_beta(e_bt)
            Q = EventFiLM.apply_film(Q0, gamma, beta)
            K = EventFiLM.apply_film(K0, gamma, beta)
            V = EventFiLM.apply_film(V0, gamma, beta)
            Q = Q / (N ** 0.5); K = K / (N ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, T, N, C)
            return self.drop(out)
        elif mode == "temporal":
            xn = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, T, C)
            e_bn = e.reshape(B * N, -1)
            Q0 = self.wq(xn)
            K0 = self.wk(xn)
            V0 = self.wv(xn)
            gamma, beta = self.film_qkv.gamma_beta(e_bn)
            Q = EventFiLM.apply_film(Q0, gamma, beta)
            K = EventFiLM.apply_film(K0, gamma, beta)
            V = EventFiLM.apply_film(V0, gamma, beta)
            Q = Q / (T ** 0.5); K = K / (T ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, N, T, C).permute(0, 2, 1, 3).contiguous()
            return self.drop(out)
        else:
            raise ValueError("mode must be 'spatial' or 'temporal'")

class STGFiLMBlock(nn.Module):
    def __init__(self, dim, event_dim, dropout=0.1):
        super().__init__()
        self.film_x = EventFiLM(event_dim, dim)
        self.att = FiLMLinearSTAttention(dim, event_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dim, dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, e):
        x = self.film_x(x, e)
        a = self.att(x, e, "spatial") + self.att(x, e, "temporal")
        x = self.norm1(x + a)
        x = self.norm2(x + self.ff(x))
        return x

class STGFiLMFormer(nn.Module):
    def __init__(self, in_dim, out_dim, horizon, dim=64, K=3, dropout=0.1, event_dim=2):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.dim = dim
        self.K = K
        self.in_proj = nn.Linear(in_dim, dim)
        self.time_embed = nn.Parameter(torch.zeros(1, HISTORY, 1, dim))
        nn.init.normal_(self.time_embed, std=0.02)
        self.block = STGFiLMBlock(dim, event_dim=event_dim, dropout=dropout)
        self.gate = nn.Linear(dim, dim)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, horizon * out_dim))

    @staticmethod
    def build_event_vector(x_hist):
        ces = x_hist[..., IDX_CES].amax(dim=1)
        est = x_hist[..., IDX_ESTOP].amax(dim=1)
        return torch.stack([ces, est], dim=-1)

    def graph_propagate(self, x, A_hat):
        outs = [x]
        xk = x
        for _ in range(self.K):
            xk = torch.matmul(A_hat, xk)
            outs.append(xk)
        return outs

    def forward(self, x_hist, A_hat):
        B, T, N, F = x_hist.shape
        e = self.build_event_vector(x_hist)
        x = self.in_proj(x_hist) + self.time_embed[:, :T]
        props = self.graph_propagate(x, A_hat)
        h = self.block(props[0], e)
        for k in range(1, len(props)):
            hk = self.block(props[k], e)
            g = torch.sigmoid(self.gate(h))
            h = h + g * hk
        h_last = h[:, -1]
        y = self.head(h_last).view(B, N, self.horizon, self.out_dim).permute(0, 2, 1, 3).contiguous()
        return y

def astgcn_support_to_adj(A_hat: torch.Tensor) -> torch.Tensor:
    """Recover a binary adjacency from the notebook support matrix for ASTGCN."""
    adj = (A_hat > 0).to(dtype=torch.float32).clone()
    adj.fill_diagonal_(0.0)
    return adj


def scaled_laplacian_torch(adj: torch.Tensor) -> torch.Tensor:
    degree = torch.diag(adj.sum(dim=1))
    laplacian = degree - adj
    lambda_max = torch.linalg.eigvals(laplacian).real.max()
    lambda_max = torch.clamp(lambda_max, min=1e-6)
    eye = torch.eye(adj.size(0), device=adj.device, dtype=adj.dtype)
    return (2.0 * laplacian / lambda_max) - eye


def cheb_polynomial_torch(l_tilde: torch.Tensor, K: int) -> List[torch.Tensor]:
    cheb = [torch.eye(l_tilde.size(0), device=l_tilde.device, dtype=l_tilde.dtype)]
    if K == 1:
        return cheb
    cheb.append(l_tilde)
    for _ in range(2, K):
        cheb.append(2.0 * l_tilde @ cheb[-1] - cheb[-2])
    return cheb


class ASTGCNTemporalAttention(nn.Module):
    def __init__(self, in_channels: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self.U1 = nn.Parameter(torch.empty(num_of_vertices))
        self.U2 = nn.Parameter(torch.empty(in_channels, num_of_vertices))
        self.U3 = nn.Parameter(torch.empty(in_channels))
        self.be = nn.Parameter(torch.empty(1, num_of_timesteps, num_of_timesteps))
        self.Ve = nn.Parameter(torch.empty(num_of_timesteps, num_of_timesteps))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.U1)
        nn.init.xavier_uniform_(self.U2)
        nn.init.uniform_(self.U3)
        nn.init.uniform_(self.be)
        nn.init.xavier_uniform_(self.Ve)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lhs = torch.einsum("bnft,n->btf", x, self.U1)
        lhs = torch.einsum("btf,fn->btn", lhs, self.U2)
        rhs = torch.einsum("f,bnft->bnt", self.U3, x)
        product = torch.matmul(lhs, rhs)
        return torch.softmax(torch.matmul(self.Ve, torch.sigmoid(product + self.be)), dim=1)


class ASTGCNSpatialAttention(nn.Module):
    def __init__(self, in_channels: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self.W1 = nn.Parameter(torch.empty(num_of_timesteps))
        self.W2 = nn.Parameter(torch.empty(in_channels, num_of_timesteps))
        self.W3 = nn.Parameter(torch.empty(in_channels))
        self.bs = nn.Parameter(torch.empty(1, num_of_vertices, num_of_vertices))
        self.Vs = nn.Parameter(torch.empty(num_of_vertices, num_of_vertices))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.W1)
        nn.init.xavier_uniform_(self.W2)
        nn.init.uniform_(self.W3)
        nn.init.uniform_(self.bs)
        nn.init.xavier_uniform_(self.Vs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lhs = torch.einsum("bnft,t->bnf", x, self.W1)
        lhs = torch.einsum("bnf,ft->bnt", lhs, self.W2)
        rhs = torch.einsum("f,bnft->bnt", self.W3, x).transpose(-1, -2)
        product = torch.matmul(lhs, rhs)
        return torch.softmax(torch.matmul(self.Vs, torch.sigmoid(product + self.bs)), dim=1)


class ASTGCNChebConvWithSAt(nn.Module):
    def __init__(self, K: int, in_channels: int, out_channels: int):
        super().__init__()
        self.K = K
        self.out_channels = out_channels
        self.Theta = nn.ParameterList([nn.Parameter(torch.empty(in_channels, out_channels)) for _ in range(K)])
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for theta in self.Theta:
            nn.init.xavier_uniform_(theta)

    def forward(self, x: torch.Tensor, spatial_attention: torch.Tensor, cheb_polynomials: List[torch.Tensor]) -> torch.Tensor:
        batch_size, num_of_vertices, _, num_of_timesteps = x.shape
        outputs = []
        for time_step in range(num_of_timesteps):
            graph_signal = x[:, :, :, time_step]
            output = torch.zeros(batch_size, num_of_vertices, self.out_channels, device=x.device, dtype=x.dtype)
            for k in range(self.K):
                T_k = cheb_polynomials[k]
                T_k_with_at = T_k.unsqueeze(0) * spatial_attention
                rhs = torch.matmul(T_k_with_at.transpose(1, 2), graph_signal)
                output = output + torch.matmul(rhs, self.Theta[k])
            outputs.append(output.unsqueeze(-1))
        return torch.relu(torch.cat(outputs, dim=-1))


class ASTGCNBlock(nn.Module):
    def __init__(self, in_channels: int, K: int, nb_chev_filter: int, nb_time_filter: int, time_strides: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self.TAt = ASTGCNTemporalAttention(in_channels, num_of_vertices, num_of_timesteps)
        self.SAt = ASTGCNSpatialAttention(in_channels, num_of_vertices, num_of_timesteps)
        self.cheb_conv_SAt = ASTGCNChebConvWithSAt(K, in_channels, nb_chev_filter)
        self.time_conv = nn.Conv2d(nb_chev_filter, nb_time_filter, kernel_size=(1, 3), stride=(1, time_strides), padding=(0, 1))
        self.residual_conv = nn.Conv2d(in_channels, nb_time_filter, kernel_size=(1, 1), stride=(1, time_strides))
        self.ln = nn.LayerNorm(nb_time_filter)

    def forward(self, x: torch.Tensor, cheb_polynomials: List[torch.Tensor]) -> torch.Tensor:
        batch_size, num_of_vertices, num_of_features, num_of_timesteps = x.shape
        temporal_attention = self.TAt(x)
        x_tat = torch.matmul(x.reshape(batch_size, -1, num_of_timesteps), temporal_attention)
        x_tat = x_tat.reshape(batch_size, num_of_vertices, num_of_features, num_of_timesteps)
        spatial_attention = self.SAt(x_tat)
        spatial_gcn = self.cheb_conv_SAt(x, spatial_attention, cheb_polynomials)
        time_conv_output = self.time_conv(spatial_gcn.permute(0, 2, 1, 3))
        x_residual = self.residual_conv(x.permute(0, 2, 1, 3))
        x_residual = torch.relu(x_residual + time_conv_output).permute(0, 3, 2, 1)
        x_residual = self.ln(x_residual).permute(0, 2, 3, 1)
        return x_residual


class ASTGCN(nn.Module):
    """Official-style recent-component ASTGCN adapted to the notebook loader."""

    def __init__(self, in_dim: int, out_dim: int, horizon: int, num_nodes: int, hidden_dim: int = HIDDEN_DIM, dropout: float = DROPOUT, K: int = 3, nb_block: int = 2, time_strides: int = 1):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.K = K
        self.blocks = nn.ModuleList([
            ASTGCNBlock(in_dim, K, hidden_dim, hidden_dim, time_strides, num_nodes, HISTORY)
        ])
        for _ in range(1, nb_block):
            self.blocks.append(ASTGCNBlock(hidden_dim, K, hidden_dim, hidden_dim, 1, num_nodes, HISTORY))
        self.final_conv = nn.Conv2d(HISTORY, horizon * out_dim, kernel_size=(1, hidden_dim))
        self._cached_cheb = None

    def forward(self, x: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        batch_size, steps, num_nodes, _ = x.shape
        if self._cached_cheb is None or self._cached_cheb[0].device != x.device:
            adj = astgcn_support_to_adj(A_hat.to(x.device))
            l_tilde = scaled_laplacian_torch(adj)
            self._cached_cheb = cheb_polynomial_torch(l_tilde, self.K)
        cheb_polynomials = self._cached_cheb
        z = x.permute(0, 2, 3, 1).contiguous()
        for block in self.blocks:
            z = block(z, cheb_polynomials)
        out = self.final_conv(z.permute(0, 3, 1, 2))[:, :, :, -1]
        out = out.permute(0, 2, 1).contiguous().view(batch_size, num_nodes, self.horizon, self.out_dim)
        return out.permute(0, 2, 1, 3).contiguous()


class DSTATemporalScaledDotProductAttention(nn.Module):
    def __init__(self, d_k: int):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, attn_mask: Optional[torch.Tensor], res_att: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k) + res_att
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        return context, scores


class DSTASpatialScaledDotProductAttention(nn.Module):
    def __init__(self, d_k: int):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q: torch.Tensor, K: torch.Tensor, attn_mask: Optional[torch.Tensor]) -> torch.Tensor:
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -1e9)
        return scores


class DSTASpatialMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int, n_heads: int):
        super().__init__()
        self.d_k = d_k
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_k * n_heads, bias=False)
        self.W_K = nn.Linear(d_model, d_k * n_heads, bias=False)

    def forward(self, input_Q: torch.Tensor, input_K: torch.Tensor, attn_mask: Optional[torch.Tensor]) -> torch.Tensor:
        batch_size = input_Q.size(0)
        Q = self.W_Q(input_Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(input_K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        if attn_mask is not None:
            attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        return DSTASpatialScaledDotProductAttention(self.d_k)(Q, K, attn_mask)


class DSTATemporalMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int, d_v: int, n_heads: int, num_of_d: int):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.n_heads = n_heads
        self.num_of_d = num_of_d
        self.W_Q = nn.Linear(d_model, d_k * n_heads, bias=False)
        self.W_K = nn.Linear(d_model, d_k * n_heads, bias=False)
        self.W_V = nn.Linear(d_model, d_v * n_heads, bias=False)
        self.fc = nn.Linear(n_heads * d_v, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, input_Q: torch.Tensor, input_K: torch.Tensor, input_V: torch.Tensor, attn_mask: Optional[torch.Tensor], res_att: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        residual = input_Q
        batch_size = input_Q.size(0)
        Q = self.W_Q(input_Q).view(batch_size, self.num_of_d, -1, self.n_heads, self.d_k).transpose(2, 3)
        K = self.W_K(input_K).view(batch_size, self.num_of_d, -1, self.n_heads, self.d_k).transpose(2, 3)
        V = self.W_V(input_V).view(batch_size, self.num_of_d, -1, self.n_heads, self.d_v).transpose(2, 3)
        if attn_mask is not None:
            attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        context, res_attn = DSTATemporalScaledDotProductAttention(self.d_k)(Q, K, V, attn_mask, res_att)
        context = context.transpose(2, 3).reshape(batch_size, self.num_of_d, -1, self.n_heads * self.d_v)
        output = self.fc(context)
        return self.norm(output + residual), res_attn


class DSTAEmbedding(nn.Module):
    def __init__(self, nb_seq: int, d_emb: int, num_of_features: int, etype: str):
        super().__init__()
        self.nb_seq = nb_seq
        self.etype = etype
        self.num_of_features = num_of_features
        self.pos_embed = nn.Embedding(nb_seq, d_emb)
        self.norm = nn.LayerNorm(d_emb)

    def forward(self, x: torch.Tensor, batch_size: int) -> torch.Tensor:
        device = x.device
        if self.etype == 'T':
            pos = torch.arange(self.nb_seq, dtype=torch.long, device=device)
            pos = pos.unsqueeze(0).unsqueeze(0).expand(batch_size, self.num_of_features, self.nb_seq)
            embedding = x.permute(0, 2, 3, 1).contiguous() + self.pos_embed(pos)
        else:
            pos = torch.arange(self.nb_seq, dtype=torch.long, device=device)
            pos = pos.unsqueeze(0).expand(batch_size, self.nb_seq)
            embedding = x + self.pos_embed(pos)
        return self.norm(embedding)


class DSTAGTU(nn.Module):
    def __init__(self, in_channels: int, time_strides: int, kernel_size: int):
        super().__init__()
        self.in_channels = in_channels
        self.conv = nn.Conv2d(in_channels, 2 * in_channels, kernel_size=(1, kernel_size), stride=(1, time_strides))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_conv = self.conv(x)
        x_p = x_conv[:, :self.in_channels, :, :]
        x_q = x_conv[:, -self.in_channels:, :, :]
        return torch.tanh(x_p) * torch.sigmoid(x_q)


class DSTAChebConvWithSAt(nn.Module):
    def __init__(self, K: int, in_channels: int, out_channels: int, num_of_vertices: int):
        super().__init__()
        self.K = K
        self.out_channels = out_channels
        self.Theta = nn.ParameterList([nn.Parameter(torch.empty(in_channels, out_channels)) for _ in range(K)])
        self.mask = nn.ParameterList([nn.Parameter(torch.empty(num_of_vertices, num_of_vertices)) for _ in range(K)])
        for theta in self.Theta:
            nn.init.xavier_uniform_(theta)
        for mask in self.mask:
            nn.init.xavier_uniform_(mask)

    def forward(self, x: torch.Tensor, spatial_attention: torch.Tensor, cheb_polynomials: List[torch.Tensor], adj_pa: torch.Tensor) -> torch.Tensor:
        batch_size, num_of_vertices, _, num_of_timesteps = x.shape
        outputs = []
        for time_step in range(num_of_timesteps):
            graph_signal = x[:, :, :, time_step]
            output = torch.zeros(batch_size, num_of_vertices, self.out_channels, device=x.device, dtype=x.dtype)
            for k in range(self.K):
                T_k = cheb_polynomials[k]
                my_spatial_attention = spatial_attention[:, k, :, :] + adj_pa * self.mask[k]
                my_spatial_attention = torch.softmax(my_spatial_attention, dim=1)
                T_k_with_at = T_k.unsqueeze(0) * my_spatial_attention
                rhs = torch.matmul(T_k_with_at.transpose(1, 2), graph_signal)
                output = output + torch.matmul(rhs, self.Theta[k])
            outputs.append(output.unsqueeze(-1))
        return torch.relu(torch.cat(outputs, dim=-1))


class DSTAGNNBlock(nn.Module):
    def __init__(self, num_of_d: int, in_channels: int, K: int, nb_chev_filter: int, nb_time_filter: int, time_strides: int, num_of_vertices: int, num_of_timesteps: int, d_model: int, d_k: int, d_v: int, n_heads: int, dropout: float):
        super().__init__()
        self.pre_conv = nn.Conv2d(num_of_timesteps, d_model, kernel_size=(1, num_of_d))
        self.embed_t = DSTAEmbedding(num_of_timesteps, num_of_vertices, num_of_d, 'T')
        self.embed_s = DSTAEmbedding(num_of_vertices, d_model, num_of_d, 'S')
        self.temporal_attn = DSTATemporalMultiHeadAttention(num_of_vertices, d_k, d_v, n_heads, num_of_d)
        self.spatial_attn = DSTASpatialMultiHeadAttention(d_model, d_k, K)
        self.cheb_conv = DSTAChebConvWithSAt(K, in_channels, nb_chev_filter, num_of_vertices)
        self.gtu3 = DSTAGTU(nb_time_filter, time_strides, 3)
        self.gtu5 = DSTAGTU(nb_time_filter, time_strides, 5)
        self.gtu7 = DSTAGTU(nb_time_filter, time_strides, 7)
        self.residual_conv = nn.Conv2d(in_channels, nb_time_filter, kernel_size=(1, 1), stride=(1, time_strides))
        self.dropout = nn.Dropout(dropout)
        self.fc_time = nn.Sequential(nn.Linear(3 * num_of_timesteps - 12, num_of_timesteps), nn.Dropout(dropout))
        self.ln = nn.LayerNorm(nb_time_filter)

    def forward(self, x: torch.Tensor, res_att: torch.Tensor, cheb_polynomials: List[torch.Tensor], adj_pa: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size, num_of_vertices, num_of_features, num_of_timesteps = x.shape
        if num_of_features == 1:
            t_emx = self.embed_t(x, batch_size)
        else:
            t_emx = x.permute(0, 2, 3, 1).contiguous()
        t_att_out, re_att = self.temporal_attn(t_emx, t_emx, t_emx, None, res_att)
        x_tat = self.pre_conv(t_att_out.permute(0, 2, 3, 1).contiguous())[:, :, :, -1].permute(0, 2, 1).contiguous()
        s_emx_tat = self.dropout(self.embed_s(x_tat, batch_size))
        s_att = self.spatial_attn(s_emx_tat, s_emx_tat, None)
        spatial_gcn = self.cheb_conv(x, s_att, cheb_polynomials, adj_pa)
        X = spatial_gcn.permute(0, 2, 1, 3).contiguous()
        x_gtu = [self.gtu3(X), self.gtu5(X), self.gtu7(X)]
        time_conv = self.fc_time(torch.cat(x_gtu, dim=-1))
        if X.shape[1] == time_conv.shape[1]:
            time_conv_output = torch.relu(X + time_conv)
        else:
            time_conv_output = torch.relu(time_conv)
        if x.shape[2] != time_conv_output.shape[1]:
            x_residual = self.residual_conv(x.permute(0, 2, 1, 3).contiguous())
        else:
            x_residual = x.permute(0, 2, 1, 3).contiguous()
        x_residual = self.ln(torch.relu(x_residual + time_conv_output).permute(0, 3, 2, 1)).permute(0, 2, 3, 1)
        return x_residual, re_att


class DSTAGNN(nn.Module):
    """Official-style DSTAGNN adapted to multi-feature railway inputs."""

    def __init__(self, in_dim: int, out_dim: int, horizon: int, num_nodes: int, nb_block: int = 4, K: int = 3, nb_chev_filter: int = 32, nb_time_filter: int = 32, time_strides: int = 1, d_model: int = 128, d_k: int = 32, d_v: int = 32, n_heads: int = 3, dropout: float = DROPOUT):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.K = K
        self._cached_cheb = None
        self.blocks = nn.ModuleList([
            DSTAGNNBlock(in_dim, in_dim, K, nb_chev_filter, nb_time_filter, time_strides, num_nodes, HISTORY, d_model, d_k, d_v, n_heads, dropout)
        ])
        for _ in range(1, nb_block):
            self.blocks.append(DSTAGNNBlock(nb_time_filter, nb_chev_filter, K, nb_chev_filter, nb_time_filter, 1, num_nodes, HISTORY, d_model, d_k, d_v, n_heads, dropout))
        self.final_conv = nn.Conv2d(HISTORY * nb_block, 128, kernel_size=(1, nb_time_filter))
        self.final_fc = nn.Linear(128, horizon * out_dim)

    def forward(self, x: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        batch_size, steps, num_nodes, _ = x.shape
        if self._cached_cheb is None or self._cached_cheb[0].device != x.device:
            adj = astgcn_support_to_adj(A_hat.to(x.device))
            l_tilde = scaled_laplacian_torch(adj)
            self._cached_cheb = cheb_polynomial_torch(l_tilde, self.K)
        adj_pa = astgcn_support_to_adj(A_hat.to(x.device))
        z = x.permute(0, 2, 3, 1).contiguous()
        need_concat = []
        res_att = 0.0
        for block in self.blocks:
            z, res_att = block(z, res_att, self._cached_cheb, adj_pa)
            if isinstance(res_att, torch.Tensor) and res_att.dim() == 5 and res_att.size(1) != 1:
                res_att = res_att.mean(dim=1, keepdim=True)
            need_concat.append(z)
        final_x = torch.cat(need_concat, dim=-1)
        out = self.final_conv(final_x.permute(0, 3, 1, 2).contiguous())[:, :, :, -1].permute(0, 2, 1).contiguous()
        out = self.final_fc(out).view(batch_size, num_nodes, self.horizon, self.out_dim)
        return out.permute(0, 2, 1, 3).contiguous()


class STAEAttentionLayer(nn.Module):
    def __init__(self, model_dim: int, num_heads: int = 4, mask: bool = False):
        super().__init__()
        if model_dim % num_heads != 0:
            raise ValueError("model_dim must be divisible by num_heads")
        self.model_dim = model_dim
        self.num_heads = num_heads
        self.mask = mask
        self.head_dim = model_dim // num_heads
        self.FC_Q = nn.Linear(model_dim, model_dim)
        self.FC_K = nn.Linear(model_dim, model_dim)
        self.FC_V = nn.Linear(model_dim, model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
        batch_size = query.shape[0]
        tgt_length = query.shape[-2]
        src_length = key.shape[-2]
        query = self.FC_Q(query)
        key = self.FC_K(key)
        value = self.FC_V(value)
        query = torch.cat(torch.split(query, self.head_dim, dim=-1), dim=0)
        key = torch.cat(torch.split(key, self.head_dim, dim=-1), dim=0)
        value = torch.cat(torch.split(value, self.head_dim, dim=-1), dim=0)
        key = key.transpose(-1, -2)
        attn_score = (query @ key) / (self.head_dim ** 0.5)
        if self.mask:
            mask = torch.ones(tgt_length, src_length, dtype=torch.bool, device=query.device).tril()
            attn_score.masked_fill_(~mask, -torch.inf)
        attn_score = torch.softmax(attn_score, dim=-1)
        out = attn_score @ value
        out = torch.cat(torch.split(out, batch_size, dim=0), dim=-1)
        return self.out_proj(out)


class STAESelfAttentionLayer(nn.Module):
    def __init__(self, model_dim: int, feed_forward_dim: int = 256, num_heads: int = 4, dropout: float = 0.1, mask: bool = False):
        super().__init__()
        self.attn = STAEAttentionLayer(model_dim, num_heads, mask)
        self.feed_forward = nn.Sequential(
            nn.Linear(model_dim, feed_forward_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feed_forward_dim, model_dim),
        )
        self.ln1 = nn.LayerNorm(model_dim)
        self.ln2 = nn.LayerNorm(model_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, dim: int = -2) -> torch.Tensor:
        x = x.transpose(dim, -2)
        residual = x
        out = self.attn(x, x, x)
        out = self.dropout1(out)
        out = self.ln1(residual + out)
        residual = out
        out = self.feed_forward(out)
        out = self.dropout2(out)
        out = self.ln2(residual + out)
        return out.transpose(dim, -2)


class STAEFormer(nn.Module):
    """Official-style STAEformer adapted to feature-rich railway inputs."""

    def __init__(self, in_dim: int, out_dim: int, horizon: int, num_nodes: int, hidden_dim: int = HIDDEN_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.num_nodes = num_nodes
        self.in_steps = HISTORY
        self.out_steps = horizon
        self.output_dim = out_dim
        self.input_dim = in_dim
        self.input_embedding_dim = 24
        self.tod_embedding_dim = 0
        self.dow_embedding_dim = 0
        self.spatial_embedding_dim = 0
        self.adaptive_embedding_dim = 80
        self.model_dim = self.input_embedding_dim + self.tod_embedding_dim + self.dow_embedding_dim + self.spatial_embedding_dim + self.adaptive_embedding_dim
        self.use_mixed_proj = True
        self.input_proj = nn.Linear(in_dim, self.input_embedding_dim)
        self.adaptive_embedding = nn.init.xavier_uniform_(nn.Parameter(torch.empty(self.in_steps, num_nodes, self.adaptive_embedding_dim)))
        self.output_proj = nn.Linear(self.in_steps * self.model_dim, self.out_steps * out_dim)
        self.attn_layers_t = nn.ModuleList([STAESelfAttentionLayer(self.model_dim, feed_forward_dim=256, num_heads=4, dropout=dropout) for _ in range(3)])
        self.attn_layers_s = nn.ModuleList([STAESelfAttentionLayer(self.model_dim, feed_forward_dim=256, num_heads=4, dropout=dropout) for _ in range(3)])

    def forward(self, x: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        features = [self.input_proj(x)]
        adp_emb = self.adaptive_embedding.expand(size=(batch_size, *self.adaptive_embedding.shape))
        features.append(adp_emb)
        z = torch.cat(features, dim=-1)
        for attn in self.attn_layers_t:
            z = attn(z, dim=1)
        for attn in self.attn_layers_s:
            z = attn(z, dim=2)
        out = z.transpose(1, 2).reshape(batch_size, self.num_nodes, self.in_steps * self.model_dim)
        out = self.output_proj(out).view(batch_size, self.num_nodes, self.out_steps, self.output_dim)
        return out.transpose(1, 2).contiguous()

def build_model(model_name: str, num_nodes: int, A_hat: Optional[torch.Tensor] = None, horizon_override: Optional[int] = None) -> nn.Module:
    horizon = HORIZON if horizon_override is None else int(horizon_override)
    if model_name == "DSTAGNN":
        return DSTAGNN(len(FEAT_COLS), len(TARGET_COLS), horizon, num_nodes=num_nodes)
    if model_name == "STGFormer":
        return STGformer(in_dim=len(FEAT_COLS), out_dim=len(TARGET_COLS), horizon=horizon,
                         dim=HIDDEN_DIM, K=3, dropout=DROPOUT)
    if model_name == "STG_MFilmFormer":
        return STGMFiLMFormer(
            in_dim=len(FEAT_COLS),
            out_dim=len(TARGET_COLS),
            horizon=horizon,
            history=HISTORY,
            ces_index=IDX_CES,
            traffic_indices=IDX_TRAFFIC,
            dim=HIDDEN_DIM,
            k_hops=3,
            dropout=DROPOUT,
            recon_lambda=0.05,
        )
    if model_name == "GCNMdynamic":
        if A_hat is None:
            raise ValueError("GCNMdynamic requires A_hat to build static supports.")
        supports = [A_hat.detach().clone(), A_hat.detach().transpose(0, 1).contiguous().clone()]
        return GCNMDynamicMulti(
            device=DEVICE,
            num_nodes=num_nodes,
            in_dim=len(FEAT_COLS),
            out_dim=len(TARGET_COLS),
            horizon=horizon,
            dropout=0.3,
            supports=supports,
            residual_channels=32,
            dilation_channels=32,
            skip_channels=256,
            end_channels=512,
            kernel_size=2,
            blocks=4,
            layers=2,
        )
    if model_name == "SBU-LSTM":
        return SBULSTM(
            input_dim=num_nodes * len(FEAT_COLS),
            hidden_dim=HIDDEN_DIM,
            out_dim=num_nodes * len(TARGET_COLS),
            horizon=horizon,
            reg_lambda=1.0,
        )
    if model_name == "GCRNODE":
        if A_hat is None:
            raise ValueError("GCRNODE requires A_hat to build the graph mask.")
        graph_mask = (A_hat.detach() > 0).to(dtype=torch.float32)
        stmg = STMemoryGraph(
            num_nodes=num_nodes,
            memory_dim=GCRNODE_CONFIG.memory_dim,
            replay_day_size=GCRNODE_CONFIG.replay_day_size,
            replay_week_size=GCRNODE_CONFIG.replay_week_size,
            topk=min(GCRNODE_CONFIG.topk, num_nodes),
            graph_mask=graph_mask,
        )
        return GCRNODEModel(
            num_nodes=num_nodes,
            input_dim=len(FEAT_COLS),
            output_dim=len(TARGET_COLS),
            hidden_dim=HIDDEN_DIM,
            horizon=horizon,
            stmg=stmg,
            depth=GCRNODE_CONFIG.diffusion_depth,
            ode_steps=GCRNODE_CONFIG.ode_steps,
            ode_solver=GCRNODE_CONFIG.ode_solver,
            kl_weight=GCRNODE_CONFIG.kl_weight,
        )
    raise ValueError(f"Unknown model: {model_name}")

# ============================================================
# 6. Training, Checkpointing, FLOPs, And Inference Timing
# ============================================================

@dataclass
class RunResult:
    dataset_name: str
    model_name: str
    run_csv: Path
    plot_path: Path
    checkpoint_path: Path
    final_metrics: MetricResult
    flops: float
    parameters: int
    inference_ms: float
    best_epoch: int


def count_parameters(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def train_one_epoch(model: nn.Module, loader: DataLoader, bundle: STGDataBundle, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    model.train()
    total_loss = 0.0
    total_samples = 0
    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        pred, y = forward_model_batch(model, batch, bundle)
        loss = loss_fn(pred, y) + model_aux_loss(model, bundle)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_size = y.shape[0]
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size
    return total_loss / max(total_samples, 1)


def metrics_to_row(prefix: str, metrics: Optional[MetricResult]) -> Dict[str, float]:
    row: Dict[str, float] = {}
    if metrics is None:
        row[f"{prefix}_loss"] = np.nan
        for h in range(1, HORIZON + 1):
            row[f"{prefix}_horizon_{h}_mae"] = np.nan
            row[f"{prefix}_horizon_{h}_rmse"] = np.nan
            row[f"{prefix}_horizon_{h}_mape"] = np.nan
        row[f"{prefix}_average_mae"] = np.nan
        row[f"{prefix}_average_rmse"] = np.nan
        row[f"{prefix}_average_mape"] = np.nan
        return row

    row[f"{prefix}_loss"] = metrics.loss
    for h in range(1, HORIZON + 1):
        row[f"{prefix}_horizon_{h}_mae"] = metrics.horizon_mae[h - 1]
        row[f"{prefix}_horizon_{h}_rmse"] = metrics.horizon_rmse[h - 1]
        row[f"{prefix}_horizon_{h}_mape"] = metrics.horizon_mape[h - 1]
    row[f"{prefix}_average_mae"] = metrics.average_mae
    row[f"{prefix}_average_rmse"] = metrics.average_rmse
    row[f"{prefix}_average_mape"] = metrics.average_mape
    return row


def measure_inference_ms(model: nn.Module, loader: DataLoader, bundle: STGDataBundle, repeats: int = 50) -> float:
    model.eval()
    batch = next(iter(loader))

    with torch.no_grad():
        for _ in range(5):
            pred, _ = forward_model_batch(model, batch, bundle)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            pred, _ = forward_model_batch(model, batch, bundle)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()
    return float((end - start) * 1000.0 / repeats)


def estimate_flops(model: nn.Module, bundle: STGDataBundle) -> float:
    """Best-effort FLOP count with model-specific dummy inputs.

    Profile a deep-copied model so THOP hooks can never contaminate the live
    training model before inference timing.
    """
    try:
        from thop import profile
    except Exception:
        return float("nan")

    num_nodes = len(bundle.ci_list)
    feat_dim = len(FEAT_COLS)
    a_hat = bundle.A_hat.detach().clone().to(DEVICE)

    try:
        prof_model = copy.deepcopy(model).to(DEVICE)
        prof_model.eval()

        if bundle.model_name == "STG_MFilmFormer":
            class ProfileWrapper(nn.Module):
                def __init__(self, wrapped: nn.Module, a_hat_buf: torch.Tensor):
                    super().__init__()
                    self.wrapped = wrapped
                    self.register_buffer("a_hat", a_hat_buf)

                def forward(self, x: torch.Tensor) -> torch.Tensor:
                    return self.wrapped(x, self.a_hat)

            wrapper = ProfileWrapper(prof_model, a_hat).to(DEVICE).eval()
            dummy_x = torch.zeros(1, HISTORY, num_nodes, feat_dim * 9, dtype=torch.float32, device=DEVICE)
            flops, _ = profile(wrapper, inputs=(dummy_x,), verbose=False)
            return float(flops)

        if bundle.model_name in GCNM_MODELS:
            hist_steps = int(getattr(GCNM_CONFIG, "history", HISTORY))
            dummy_stat = torch.zeros(1, 8, hist_steps, num_nodes, feat_dim, dtype=torch.float32, device=DEVICE)
            hist_len = int((GCNM_CONFIG.nh + GCNM_CONFIG.nd + GCNM_CONFIG.nw) * (GCNM_CONFIG.tau or GCNM_CONFIG.history))
            if hist_len <= 0:
                hist_len = 1
            dummy_hist = torch.zeros(1, hist_len, hist_steps, num_nodes, feat_dim, dtype=torch.float32, device=DEVICE)
            flops, _ = profile(prof_model, inputs=(dummy_stat, dummy_hist), verbose=False)
            return float(flops)

        if bundle.model_name in SBU_MODELS:
            hist_steps = int(getattr(SBU_CONFIG, "history", HISTORY))
            flat_dim = num_nodes * feat_dim
            dummy_x = torch.zeros(1, hist_steps, flat_dim, dtype=torch.float32, device=DEVICE)
            dummy_mask = torch.ones_like(dummy_x)
            dummy_true = torch.zeros_like(dummy_x)
            flops, _ = profile(prof_model, inputs=(dummy_x, dummy_mask, dummy_true), verbose=False)
            return float(flops)

        if bundle.model_name in GCRNODE_MODELS:
            hist_steps = int(getattr(GCRNODE_CONFIG, "history", HISTORY))
            model_horizon = int(getattr(prof_model, "horizon", HORIZON))
            dummy_x = torch.zeros(1, hist_steps, num_nodes, feat_dim, dtype=torch.float32, device=DEVICE)
            dummy_mask = torch.ones_like(dummy_x)
            dummy_hist_idx = torch.arange(hist_steps, device=DEVICE, dtype=torch.long).unsqueeze(0)
            dummy_fut_idx = torch.arange(model_horizon, device=DEVICE, dtype=torch.long).unsqueeze(0)
            flops, _ = profile(prof_model, inputs=(dummy_x, dummy_mask, dummy_hist_idx, dummy_fut_idx), verbose=False)
            return float(flops)

        class ProfileWrapper(nn.Module):
            def __init__(self, wrapped: nn.Module, a_hat_buf: torch.Tensor):
                super().__init__()
                self.wrapped = wrapped
                self.register_buffer("a_hat", a_hat_buf)

            def forward(self, x: torch.Tensor) -> torch.Tensor:
                return self.wrapped(x, self.a_hat)

        wrapper = ProfileWrapper(prof_model, a_hat).to(DEVICE).eval()
        dummy = torch.zeros(1, HISTORY, num_nodes, feat_dim, dtype=torch.float32, device=DEVICE)
        flops, _ = profile(wrapper, inputs=(dummy,), verbose=False)
        return float(flops)
    except Exception as exc:
        print(f"FLOP profiling skipped: {exc}")
        return float("nan")


def save_checkpoint(
    path: Path,
    model: nn.Module,
    bundle: STGDataBundle,
    dataset_name: str,
    model_name: str,
    best_epoch: int,
    horizon_override: Optional[int] = None,
    extra_metadata: Optional[Dict[str, object]] = None,
) -> None:
    payload = {
        "dataset": dataset_name,
        "model": model_name,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
        "history": HISTORY,
        "horizon": HORIZON if horizon_override is None else int(horizon_override),
        "feature_cols": FEAT_COLS,
        "target_cols": TARGET_COLS,
        "use_log1p": bundle.use_log1p,
        "use_zscore": bundle.use_zscore,
    }
    if extra_metadata:
        payload.update(extra_metadata)
    torch.save(payload, path)



def format_gflops(flops: float) -> str:
    """Human-readable FLOP display; CSVs still store the exact raw FLOP count."""
    if not np.isfinite(flops):
        return "NaN GFLOPs"
    return f"{flops / 1e9:.3f} GFLOPs"


def format_params_m(params: int) -> str:
    """Human-readable parameter display; CSVs still store the exact integer count."""
    if pd.isna(params):
        return "NaN M"
    return f"{params / 1e6:.3f} M"


def require_benchmark_setup() -> None:
    """Fail early if notebook cells were run out of order."""
    required = [
        "load_stg_dataset",
        "make_loaders",
        "evaluate_model",
        "build_model",
        "upsert_efficiency_row",
        "upsert_new_error_rows",
        "plot_run_metrics",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Notebook setup is incomplete. Please run cells 2 through 7 in order "
            "before launching a training run. Missing definitions: " + ", ".join(missing)
        )

def train_experiment(dataset_name: str, model_name: str, epochs: int = EPOCHS) -> RunResult:
    require_benchmark_setup()
    print(f"\n=== Training {model_name} on {dataset_name} ===")
    bundle = load_stg_dataset(dataset_name, model_name=model_name)
    train_loader, val_loader, test_loader, simulation_loader = make_loaders(bundle)

    model = build_model(model_name, num_nodes=len(bundle.ci_list), A_hat=bundle.A_hat).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss()

    run_csv = RUNS_DIR / f"{dataset_name}_{model_name}_metrics.csv"
    checkpoint_path = MODELS_DIR / f"{dataset_name}_{model_name}_best.pt"

    rows = []
    best_selection_loss = float("inf")
    best_epoch = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, bundle, optimizer, loss_fn)
        val_metrics = evaluate_model(model, val_loader, bundle, loss_fn) if val_loader is not None else None

        selection_loss = val_metrics.loss if val_metrics is not None else train_loss
        if selection_loss < best_selection_loss:
            best_selection_loss = selection_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        row = {
            "dataset": dataset_name,
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "selection_loss": selection_loss,
            "is_best_so_far": epoch == best_epoch,
        }
        row.update(metrics_to_row("val", val_metrics))
        row.update(metrics_to_row("test", None))
        rows.append(row)

        val_text = "no validation" if val_metrics is None else f"val_loss={val_metrics.loss:.5f}"
        print(
            f"epoch {epoch:03d} | train_loss={train_loss:.5f} | "
            f"{val_text}"
        )

        pd.DataFrame(rows).to_csv(run_csv, index=False)

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    final_metrics = evaluate_model(model, test_loader, bundle, loss_fn)
    save_checkpoint(checkpoint_path, model, bundle, dataset_name, model_name, best_epoch)

    params = count_parameters(model)
    flops = estimate_flops(model, bundle)
    inference_ms = measure_inference_ms(model, test_loader, bundle)

    upsert_efficiency_row(dataset_name, model_name, flops, params, inference_ms, final_metrics.average_mae)
    grouped_rows = evaluate_model_grouped(model, [test_loader, simulation_loader], bundle)
    upsert_new_error_rows(dataset_name, model_name, grouped_rows)

    plot_path = plot_run_metrics(run_csv)

    print(
        f"Final test average | MAE={final_metrics.average_mae:.4f} "
        f"RMSE={final_metrics.average_rmse:.4f} MAPE={final_metrics.average_mape:.2f}%"
    )
    print(
        f"Efficiency | FLOPs={format_gflops(flops)} | "
        f"Params={format_params_m(params)} | "
        f"Inference={inference_ms:.3f} ms / prediction round"
    )
    print(f"Saved run metrics: {run_csv}")
    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Updated result tables: {EFFICIENCY_CSV}, {NEW_ERROR_KPI_CSV}")

    return RunResult(
        dataset_name=dataset_name,
        model_name=model_name,
        run_csv=run_csv,
        plot_path=plot_path,
        checkpoint_path=checkpoint_path,
        final_metrics=final_metrics,
        flops=flops,
        parameters=params,
        inference_ms=inference_ms,
        best_epoch=best_epoch,
    )





### plots definitions


In [4]:
from __future__ import annotations

# ============================================================
# Round 1. Plot Training Loss Curves From Saved CSVs
# ============================================================

PRIMARY_MODELS = ["STGFormer", "STG_MFilmFormer", "GCNMdynamic", "SBU-LSTM", "GCRNODE"]
ADDITIONAL_MODELS = ["DSTAGNN"]


def read_run_curve(dataset_name: str, model_name: str) -> Optional[pd.DataFrame]:
    """Read one saved run metric CSV. Missing files simply mean that model is not trained yet."""
    run_csv = RUNS_DIR / f"{dataset_name}_{model_name}_metrics.csv"
    if not run_csv.exists():
        return None
    df = pd.read_csv(run_csv)
    if df.empty:
        return None
    return df


def plot_dataset_training_loss_curves(dataset_name: str) -> List[Path]:
    """Create one training-loss line plot over epochs for one dataset.

    Each plot compares all models that already have saved run CSVs.
    Test-set errors are not plotted here; they are stored in training_kpi_summary.csv.
    """
    model_colors = {
        "DSTAGNN": "#d62728",
        "STGFormer": "#1f77b4",
        "STG_MFilmFormer": "#2ca02c",
        "GCNMdynamic": "#ff7f0e",
        "SBU-LSTM": "#8c564b",
        "GCRNODE": "#9467bd",
    }

    available = {}
    for model_name in MODEL_ORDER:
        df = read_run_curve(dataset_name, model_name)
        if df is not None:
            available[model_name] = df

    if not available:
        print(f"No saved run metric CSVs found for {dataset_name}; no plot generated.")
        return []

    fig, ax = plt.subplots(figsize=(11, 6))
    plotted = 0
    for model_name in MODEL_ORDER:
        df = available.get(model_name)
        if df is None or "train_loss" not in df.columns:
            continue
        curve = df[["epoch", "train_loss"]].dropna()
        if curve.empty:
            continue
        ax.plot(
            curve["epoch"],
            curve["train_loss"],
            label=model_name,
            linewidth=2.2 if model_name == "STG_MFilmFormer" else 1.8,
            marker="o" if len(curve) <= 25 else None,
            markersize=3.5,
            color=model_colors.get(model_name),
        )
        plotted += 1

    if plotted == 0:
        plt.close(fig)
        print(f"No train_loss curves found for {dataset_name}; no plot generated.")
        return []

    ax.set_title(f"{dataset_name} - Training Loss", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss (SmoothL1 on transformed targets)")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, frameon=True)
    fig.tight_layout()

    path = PLOTS_DIR / f"{dataset_name}_training_loss_curves.png"
    fig.savefig(path, dpi=180)
    plt.close(fig)

    print(f"Generated training-loss plot for {dataset_name}: {path}")
    return [path]


def plot_all_training_loss_curves() -> List[Path]:
    """Regenerate all training-loss curves from saved CSVs without retraining."""
    all_paths = []
    for dataset_name in DATASETS:
        all_paths.extend(plot_dataset_training_loss_curves(dataset_name))
    print(f"Generated {len(all_paths)} plot(s) total.")
    return all_paths


# Backward-compatible names used by train_experiment and older cells.
def plot_dataset_training_error_curves(dataset_name: str) -> List[Path]:
    return plot_dataset_training_loss_curves(dataset_name)


def plot_all_training_error_curves() -> List[Path]:
    return plot_all_training_loss_curves()


def plot_run_metrics(run_csv: Path) -> Path:
    df = pd.read_csv(run_csv, nrows=1)
    dataset_name = str(df["dataset"].iloc[0])
    paths = plot_dataset_training_loss_curves(dataset_name)
    return paths[0] if paths else PLOTS_DIR / "no_plot_generated.png"


def plot_all_existing_runs() -> List[Path]:
    return plot_all_training_error_curves()


def plot_all_dataset_metric_comparisons() -> List[Path]:
    return plot_all_training_error_curves()



## trainings round 1 (complete data)


In [5]:
from __future__ import annotations

# ============================================================
# Round 1. Train Complete-Data Benchmark (All 6 Models)
# ============================================================

# This round trains the complete-data benchmark models and updates the standard
# summary CSVs in forecasting_results_missingData/.
# Models in this round:
# - DSTAGNN
# - STGFormer
# - STG_MFilmFormer
# - GCNMdynamic
# - SBU-LSTM
# - GCRNODE

RUN_ROUND1_COMPLETE_DATA = False
EPOCHS_TO_RUN = EPOCHS

Enable_datasetA = False
Enable_datasetB = False
Enable_datasetC = True

Enable_DSTAGNN = False
Enable_STGFormer = False
Enable_STGMFiLMFormer = True
Enable_GCNMdynamic = False
Enable_SBULSTM = False
Enable_GCRNODE = False

DATASET_ENABLES = {
    "STGdata_A": Enable_datasetA,
    "STGdata_B": Enable_datasetB,
    "STGdata_C": Enable_datasetC,
}
ROUND1_MODEL_ENABLES = {
    "DSTAGNN": Enable_DSTAGNN,
    "STGFormer": Enable_STGFormer,
    "STG_MFilmFormer": Enable_STGMFiLMFormer,
    "GCNMdynamic": Enable_GCNMdynamic,
    "SBU-LSTM": Enable_SBULSTM,
    "GCRNODE": Enable_GCRNODE,
}

ROUND1_DATASETS_TO_RUN = [name for name, enabled in DATASET_ENABLES.items() if enabled]
ROUND1_MODELS_TO_RUN = [name for name, enabled in ROUND1_MODEL_ENABLES.items() if enabled]

if RUN_ROUND1_COMPLETE_DATA:
    if "require_benchmark_setup" not in globals():
        raise RuntimeError("Please run cells 2 through 9 in order before launching Round 1.")
    require_benchmark_setup()
    print(f"Round 1 datasets: {ROUND1_DATASETS_TO_RUN}")
    print(f"Round 1 complete-data models: {ROUND1_MODELS_TO_RUN}")
    round1_results = []
    for dataset_name in ROUND1_DATASETS_TO_RUN:
        for model_name in ROUND1_MODELS_TO_RUN:
            round1_results.append(train_experiment(dataset_name, model_name, epochs=EPOCHS_TO_RUN))
    plot_all_training_loss_curves()
    print("Round 1 complete-data training finished.")
else:
    print("Round 1 disabled. Set RUN_ROUND1_COMPLETE_DATA = True and enable the datasets/models you want to train.")



Round 1 disabled. Set RUN_ROUND1_COMPLETE_DATA = True and enable the datasets/models you want to train.


## training round 2 (missing-aware models only )


In [6]:
from __future__ import annotations

# ============================================================
# Round 2 Functions. Scenario-Specific Training And Evaluation
# ============================================================


def scenario_ratio_percent(missing_ratio: float) -> int:
    return int(round(float(missing_ratio) * 100.0))


def scenario_artifact_tag(scenario_name: str, missing_ratio: float) -> str:
    return f"{scenario_name}_{scenario_ratio_percent(missing_ratio)}pct"


def scenario_run_csv_path(dataset_name: str, model_name: str, scenario_name: str, missing_ratio: float) -> Path:
    scenario_dir = RUNS_DIR / scenario_name
    scenario_dir.mkdir(parents=True, exist_ok=True)
    return scenario_dir / f"{dataset_name}_{model_name}_{scenario_artifact_tag(scenario_name, missing_ratio)}_metrics.csv"


def scenario_checkpoint_path(dataset_name: str, model_name: str, scenario_name: str, missing_ratio: float) -> Path:
    scenario_dir = MODELS_DIR / scenario_name
    scenario_dir.mkdir(parents=True, exist_ok=True)
    return scenario_dir / f"{dataset_name}_{model_name}_{scenario_artifact_tag(scenario_name, missing_ratio)}_best.pt"


def missing_case_kpi_csv_path(scenario_name: str, missing_ratio: float) -> Path:
    case_tag = scenario_artifact_tag(scenario_name, missing_ratio).lower()
    return MISSING_SCENARIO_DIR / f"{case_tag}_grouped_kpi.csv"


def missing_rate_to_pattern(scenario_name: str, missing_ratio: float) -> MissingPatternConfig:
    base = MISSING_SCENARIOS[scenario_name]
    return replace(
        base,
        enabled=True,
        pattern=scenario_name,
        missing_ratio=missing_ratio,
        observed_ratio=1.0 - missing_ratio,
    )


def make_missing_scenario_loaders(bundle: STGDataBundle, pattern_cfg: MissingPatternConfig):
    split = bundle.split
    if bundle.model_name == "STG_MFilmFormer":
        cfg = replace(MFILM_CONFIG)
        cfg.horizon = MISSING_EVAL_HORIZON
        apply_missing_pattern(cfg, pattern_cfg)
        return build_mfilm_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            cfg,
        )
    if bundle.model_name in GCNM_MODELS:
        cfg = replace(GCNM_CONFIG)
        cfg.horizon = MISSING_EVAL_HORIZON
        apply_missing_pattern(cfg, pattern_cfg)
        return build_gcnm_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            cfg,
        )
    if bundle.model_name in SBU_MODELS:
        cfg = replace(SBU_CONFIG)
        cfg.horizon = MISSING_EVAL_HORIZON
        apply_missing_pattern(cfg, pattern_cfg)
        return build_sbu_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            cfg,
        )
    if bundle.model_name in GCRNODE_MODELS:
        cfg = replace(GCRNODE_CONFIG)
        cfg.horizon = MISSING_EVAL_HORIZON
        apply_missing_pattern(cfg, pattern_cfg)
        return build_gcrnode_loaders(
            bundle.X.detach().cpu().numpy(),
            bundle.Y.detach().cpu().numpy(),
            bundle.graph_dists,
            split,
            BATCH_SIZE,
            cfg,
        )
    cfg = replace(SBU_CONFIG)
    cfg.horizon = MISSING_EVAL_HORIZON
    apply_missing_pattern(cfg, pattern_cfg)
    return build_masked_loaders(
        bundle.X.detach().cpu().numpy(),
        bundle.Y.detach().cpu().numpy(),
        bundle.graph_dists,
        split,
        BATCH_SIZE,
        cfg,
    )


def evaluate_model_dynamic(model: nn.Module, loader: DataLoader, bundle: STGDataBundle, loss_fn: nn.Module) -> MetricResult:
    model.eval()
    total_loss = 0.0
    total_samples = 0
    true_by_horizon: Optional[List[List[np.ndarray]]] = None
    pred_by_horizon: Optional[List[List[np.ndarray]]] = None

    for batch in loader:
        pred, y = forward_model_batch(model, batch, bundle)
        loss = loss_fn(pred, y) + model_aux_loss(model, bundle)
        batch_size = y.shape[0]
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size

        y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
        y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()
        horizon_count = y_true.shape[1]
        if true_by_horizon is None:
            true_by_horizon = [[] for _ in range(horizon_count)]
            pred_by_horizon = [[] for _ in range(horizon_count)]
        for h in range(horizon_count):
            true_by_horizon[h].append(y_true[:, h])
            pred_by_horizon[h].append(y_pred[:, h])

    if true_by_horizon is None or pred_by_horizon is None:
        return MetricResult(0.0, np.array([]), np.array([]), np.array([]), float('nan'), float('nan'), float('nan'))

    horizon_count = len(true_by_horizon)
    horizon_mae = np.zeros(horizon_count, dtype=np.float64)
    horizon_rmse = np.zeros(horizon_count, dtype=np.float64)
    horizon_mape = np.zeros(horizon_count, dtype=np.float64)
    for h in range(horizon_count):
        yt = np.concatenate(true_by_horizon[h], axis=0)
        yp = np.concatenate(pred_by_horizon[h], axis=0)
        horizon_mae[h], horizon_rmse[h], horizon_mape[h] = metric_triplet_np(yt, yp)
    return MetricResult(
        loss=total_loss / max(total_samples, 1),
        horizon_mae=horizon_mae,
        horizon_rmse=horizon_rmse,
        horizon_mape=horizon_mape,
        average_mae=float(np.nanmean(horizon_mae)),
        average_rmse=float(np.nanmean(horizon_rmse)),
        average_mape=float(np.nanmean(horizon_mape)),
    )


def metrics_to_row_dynamic(prefix: str, metrics: Optional[MetricResult]) -> Dict[str, float]:
    row: Dict[str, float] = {}
    if metrics is None:
        row[f"{prefix}_loss"] = np.nan
        return row
    row[f"{prefix}_loss"] = metrics.loss
    for h in range(1, len(metrics.horizon_mae) + 1):
        row[f"{prefix}_horizon_{h}_mae"] = metrics.horizon_mae[h - 1]
        row[f"{prefix}_horizon_{h}_rmse"] = metrics.horizon_rmse[h - 1]
        row[f"{prefix}_horizon_{h}_mape"] = metrics.horizon_mape[h - 1]
    row[f"{prefix}_average_mae"] = metrics.average_mae
    row[f"{prefix}_average_rmse"] = metrics.average_rmse
    row[f"{prefix}_average_mape"] = metrics.average_mape
    return row


def train_missing_scenario_experiment(
    dataset_name: str,
    model_name: str,
    scenario_name: str,
    missing_ratio: float,
    epochs: int = EPOCHS,
) -> RunResult:
    require_benchmark_setup()
    tag = scenario_artifact_tag(scenario_name, missing_ratio)
    print(f"\n=== Round 2 Training {model_name} on {dataset_name} [{tag}] ===")
    bundle = load_stg_dataset(dataset_name, model_name=model_name)
    pattern_cfg = missing_rate_to_pattern(scenario_name, missing_ratio)
    train_loader, val_loader, test_loader, simulation_loader = make_missing_scenario_loaders(bundle, pattern_cfg)

    model = build_model(model_name, num_nodes=len(bundle.ci_list), A_hat=bundle.A_hat, horizon_override=MISSING_EVAL_HORIZON).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss()

    run_csv = scenario_run_csv_path(dataset_name, model_name, scenario_name, missing_ratio)
    checkpoint_path = scenario_checkpoint_path(dataset_name, model_name, scenario_name, missing_ratio)

    rows = []
    best_selection_loss = float('inf')
    best_epoch = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, bundle, optimizer, loss_fn)
        val_metrics = evaluate_model_dynamic(model, val_loader, bundle, loss_fn) if val_loader is not None else None
        selection_loss = val_metrics.loss if val_metrics is not None else train_loss
        if selection_loss < best_selection_loss:
            best_selection_loss = selection_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        row = {
            'dataset': dataset_name,
            'model': model_name,
            'scenario': scenario_name,
            'missing_ratio': missing_ratio,
            'artifact_tag': tag,
            'epoch': epoch,
            'train_loss': train_loss,
            'selection_loss': selection_loss,
            'is_best_so_far': epoch == best_epoch,
        }
        row.update(metrics_to_row_dynamic('val', val_metrics))
        rows.append(row)
        pd.DataFrame(rows).to_csv(run_csv, index=False)
        val_text = 'no validation' if val_metrics is None else f"val_loss={val_metrics.loss:.5f}"
        print(f"epoch {epoch:03d} | train_loss={train_loss:.5f} | {val_text}")

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    final_metrics = evaluate_model_dynamic(model, test_loader, bundle, loss_fn)
    save_checkpoint(
        checkpoint_path,
        model,
        bundle,
        dataset_name,
        model_name,
        best_epoch,
        horizon_override=MISSING_EVAL_HORIZON,
        extra_metadata={'scenario': scenario_name, 'missing_ratio': missing_ratio, 'artifact_tag': tag},
    )

    params = count_parameters(model)
    flops = estimate_flops(model, bundle)
    inference_ms = measure_inference_ms(model, test_loader, bundle)
    plot_path = plot_run_metrics(run_csv)

    print(
        f"Round 2 final test average | MAE={final_metrics.average_mae:.4f} "
        f"RMSE={final_metrics.average_rmse:.4f} MAPE={final_metrics.average_mape:.2f}%"
    )
    print(f"Saved Round 2 run metrics: {run_csv}")
    print(f"Saved Round 2 checkpoint: {checkpoint_path}")

    return RunResult(
        dataset_name=dataset_name,
        model_name=model_name,
        run_csv=run_csv,
        plot_path=plot_path,
        checkpoint_path=checkpoint_path,
        final_metrics=final_metrics,
        flops=flops,
        parameters=params,
        inference_ms=inference_ms,
        best_epoch=best_epoch,
    )


def missing_metric_triplet(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> Dict[str, float]:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    denom = np.sum(np.abs(yt))
    wape = float('nan') if denom <= eps else float(np.sum(np.abs(yp - yt)) / denom * 100.0)
    return {
        'mae': float(mean_absolute_error(yt, yp)),
        'rmse': float(mean_squared_error(yt, yp, squared=False)),
        'wape': wape,
    }


@torch.no_grad()
def evaluate_missing_scenario_model(model: nn.Module, loader: DataLoader, bundle: STGDataBundle) -> Dict[str, float]:
    model.eval()
    true_parts: List[np.ndarray] = []
    pred_parts: List[np.ndarray] = []
    for batch in loader:
        pred, y = forward_model_batch(model, batch, bundle)
        y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
        y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()
        true_parts.append(y_true)
        pred_parts.append(y_pred)
    if not true_parts:
        return {'mae': float('nan'), 'rmse': float('nan'), 'wape': float('nan')}
    return missing_metric_triplet(np.concatenate(true_parts, axis=0), np.concatenate(pred_parts, axis=0))


@torch.no_grad()
def evaluate_missing_scenario_model_grouped(model: nn.Module, loaders: List[DataLoader], bundle: STGDataBundle) -> List[Dict[str, object]]:
    model.eval()
    horizon_count = MISSING_EVAL_HORIZON
    true_by_horizon: List[List[np.ndarray]] = [[] for _ in range(horizon_count)]
    pred_by_horizon: List[List[np.ndarray]] = [[] for _ in range(horizon_count)]

    for loader in loaders:
        for batch in loader:
            pred, y = forward_model_batch(model, batch, bundle)
            y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
            y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()
            for h in range(horizon_count):
                true_by_horizon[h].append(y_true[:, h])
                pred_by_horizon[h].append(y_pred[:, h])

    per_group_per_horizon: Dict[str, Dict[str, List[float]]] = {}
    for group_name in list(NEW_ERROR_TARGET_GROUPS.keys()) + [NEW_ERROR_BALANCED_GROUP]:
        per_group_per_horizon[group_name] = {m: [] for m in NEW_ERROR_METRICS}

    for h in range(horizon_count):
        yt_all = np.concatenate(true_by_horizon[h], axis=0)
        yp_all = np.concatenate(pred_by_horizon[h], axis=0)
        group_metrics: Dict[str, Dict[str, float]] = {}
        for group_name, group_cols in NEW_ERROR_TARGET_GROUPS.items():
            idxs = [TARGET_COLS.index(col) for col in group_cols]
            group_metrics[group_name] = metric_five_np(yt_all[..., idxs], yp_all[..., idxs])
            for metric_name, metric_value in group_metrics[group_name].items():
                per_group_per_horizon[group_name][metric_name].append(metric_value)
        balanced_metrics = metric_five_np(yt_all, yp_all)
        for metric_name, metric_value in balanced_metrics.items():
            per_group_per_horizon[NEW_ERROR_BALANCED_GROUP][metric_name].append(metric_value)

    rows: List[Dict[str, object]] = []
    for group_name, metric_dict in per_group_per_horizon.items():
        row = {'target_group': group_name}
        for h in range(1, horizon_count + 1):
            for metric_name in NEW_ERROR_METRICS:
                row[f'horizon_{h}_{metric_name}'] = metric_dict[metric_name][h - 1]
        for metric_name in NEW_ERROR_METRICS:
            row[f'average_{metric_name}'] = float(np.nanmean(metric_dict[metric_name]))
        rows.append(row)
    return rows


def update_missing_scenario_csv_for_case(
    scenario_name: str,
    missing_ratio: float,
    selected_datasets: Optional[List[str]] = None,
    selected_models: Optional[List[str]] = None,
) -> pd.DataFrame:
    require_benchmark_setup()
    datasets = list(DATASETS.keys()) if selected_datasets is None else selected_datasets
    models = MISSING_SCENARIO_MODELS if selected_models is None else selected_models
    pattern_cfg = missing_rate_to_pattern(scenario_name, missing_ratio)
    output_path = missing_case_kpi_csv_path(scenario_name, missing_ratio)
    preferred = ['traffic_only', 'emergency_only', NEW_ERROR_BALANCED_GROUP]

    existing_df = pd.read_csv(output_path) if output_path.exists() else pd.DataFrame()
    rows: List[Dict[str, object]] = []
    for dataset_name in datasets:
        for model_name in models:
            checkpoint_path = scenario_checkpoint_path(dataset_name, model_name, scenario_name, missing_ratio)
            if not checkpoint_path.exists():
                print(f"Missing scenario-specific checkpoint, keeping existing rows if any: {checkpoint_path.name}")
                continue

            bundle = load_stg_dataset(dataset_name, model_name=model_name)
            _, _, test_loader, simulation_loader = make_missing_scenario_loaders(bundle, pattern_cfg)
            payload = torch.load(checkpoint_path, map_location=DEVICE)
            state_dict = payload['state_dict'] if isinstance(payload, dict) and 'state_dict' in payload else payload
            model = build_model(model_name, num_nodes=len(bundle.ci_list), A_hat=bundle.A_hat, horizon_override=MISSING_EVAL_HORIZON).to(DEVICE)
            try:
                model.load_state_dict(state_dict, strict=True)
            except RuntimeError as exc:
                print(f"Skipping incompatible scenario checkpoint and keeping existing rows: {checkpoint_path.name} ({exc})")
                continue
            grouped_rows = evaluate_missing_scenario_model_grouped(model, [test_loader, simulation_loader], bundle)
            for row in grouped_rows:
                row['dataset'] = dataset_name
                row['model'] = model_name
                row['scenario'] = scenario_name
                row['missing_ratio'] = missing_ratio
                rows.append(row)

    new_df = pd.DataFrame(rows)
    key_cols = ['dataset', 'model', 'scenario', 'missing_ratio', 'target_group']
    if existing_df.empty:
        df = new_df.copy()
    elif new_df.empty:
        df = existing_df.copy()
    else:
        existing_keys = set(existing_df.columns)
        for col in existing_df.columns:
            if col not in new_df.columns:
                new_df[col] = np.nan
        for col in new_df.columns:
            if col not in existing_df.columns:
                existing_df[col] = np.nan
        existing_df = existing_df[new_df.columns]
        replace_keys = set(tuple(row) for row in new_df[key_cols].itertuples(index=False, name=None))
        keep_mask = [tuple(row) not in replace_keys for row in existing_df[key_cols].itertuples(index=False, name=None)]
        df = pd.concat([existing_df.loc[keep_mask], new_df], ignore_index=True)

    if not df.empty:
        df['target_group'] = pd.Categorical(df['target_group'], categories=preferred, ordered=True)
        ordered_cols = ['dataset', 'model', 'scenario', 'missing_ratio', 'target_group'] + [
            c for c in df.columns if c not in {'dataset', 'model', 'scenario', 'missing_ratio', 'target_group'}
        ]
        df = df[ordered_cols].sort_values(['dataset', 'model', 'target_group'], kind='stable').reset_index(drop=True)
    df.to_csv(output_path, index=False)
    print(f"Updated missing-case grouped KPI CSV without overwriting other rows: {output_path}")
    return df


def run_missing_scenario_sweep_from_artifacts(
    scenario_name: str,
    selected_ratios: Optional[List[float]] = None,
    selected_datasets: Optional[List[str]] = None,
    selected_models: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    require_benchmark_setup()
    ratios = MISSING_RATIO_GRID if selected_ratios is None else selected_ratios
    outputs: Dict[str, pd.DataFrame] = {}
    for missing_ratio in ratios:
        outputs[scenario_artifact_tag(scenario_name, missing_ratio)] = update_missing_scenario_csv_for_case(
            scenario_name,
            missing_ratio,
            selected_datasets=selected_datasets,
            selected_models=selected_models,
        )
    return outputs






In [7]:
from __future__ import annotations

# ============================================================
# Round 2. Train And Evaluate One Missingness Case (4-Step KPI)
# ============================================================

# Configure one case here, then rerun this cell manually for the next case.
# Example workflow:
# - set SELECTED_SCENARIO = "RM" and SELECTED_MISSING_RATIO = 0.2, run cell
# - change to 0.4, run cell again
# - change to "TM", run cell again

RUN_ROUND2_MISSINGNESS = False
ROUND2_EPOCHS_TO_RUN = EPOCHS

SELECTED_SCENARIO = 'RM'   # one of: 'RM', 'TM', 'SM', 'STM'
SELECTED_MISSING_RATIO = 0.4

Enable_datasetA = False
Enable_datasetB = False
Enable_datasetC = True

Enable_STGMFiLMFormer = True
Enable_GCNMdynamic = False
Enable_SBULSTM = False
Enable_GCRNODE = False

ROUND2_DATASET_ENABLES = {
    'STGdata_A': Enable_datasetA,
    'STGdata_B': Enable_datasetB,
    'STGdata_C': Enable_datasetC,
}
ROUND2_MODEL_ENABLES = {
    'STG_MFilmFormer': Enable_STGMFiLMFormer,
    'GCNMdynamic': Enable_GCNMdynamic,
    'SBU-LSTM': Enable_SBULSTM,
    'GCRNODE': Enable_GCRNODE,
}

ROUND2_DATASETS_TO_RUN = [name for name, enabled in ROUND2_DATASET_ENABLES.items() if enabled]
ROUND2_MODELS_TO_RUN = [name for name, enabled in ROUND2_MODEL_ENABLES.items() if enabled]

if RUN_ROUND2_MISSINGNESS:
    if 'train_missing_scenario_experiment' not in globals():
        raise RuntimeError('Please run the Round 2 function cell before launching Round 2.')
    if SELECTED_SCENARIO not in {'RM', 'TM', 'SM', 'STM'}:
        raise RuntimeError(f'Unsupported Round 2 scenario: {SELECTED_SCENARIO}')
    if SELECTED_MISSING_RATIO not in {0.2, 0.4, 0.6, 0.8}:
        raise RuntimeError('SELECTED_MISSING_RATIO must be one of 0.2, 0.4, 0.6, 0.8')
    if not ROUND2_DATASETS_TO_RUN or not ROUND2_MODELS_TO_RUN:
        raise RuntimeError('Round 2 needs at least one enabled dataset and one enabled model.')

    print(f'Round 2 scenario: {SELECTED_SCENARIO}')
    print(f'Round 2 missing ratio: {SELECTED_MISSING_RATIO}')
    print(f'Round 2 datasets: {ROUND2_DATASETS_TO_RUN}')
    print(f'Round 2 models: {ROUND2_MODELS_TO_RUN}')

    round2_case_results = []
    for dataset_name in ROUND2_DATASETS_TO_RUN:
        for model_name in ROUND2_MODELS_TO_RUN:
            round2_case_results.append(
                train_missing_scenario_experiment(
                    dataset_name,
                    model_name,
                    SELECTED_SCENARIO,
                    SELECTED_MISSING_RATIO,
                    epochs=ROUND2_EPOCHS_TO_RUN,
                )
            )

    round2_case_csv = update_missing_scenario_csv_for_case(
        SELECTED_SCENARIO,
        SELECTED_MISSING_RATIO,
        selected_datasets=ROUND2_DATASETS_TO_RUN,
        selected_models=ROUND2_MODELS_TO_RUN,
    )
    display(round2_case_csv)
    print('Round 2 single-case training and evaluation finished.')
else:
    print('Round 2 disabled. Set RUN_ROUND2_MISSINGNESS = True, choose one scenario and one missing ratio, then rerun this cell for each case you want.')



Round 2 disabled. Set RUN_ROUND2_MISSINGNESS = True, choose one scenario and one missing ratio, then rerun this cell for each case you want.


### Round 2 Quick Efficiency print


In [10]:
from __future__ import annotations

# ============================================================
# Round 2 Quick Efficiency Check (No CSV Write)
# ============================================================

RUN_ROUND2_EFFICIENCY_CHECK = True

if RUN_ROUND2_EFFICIENCY_CHECK:
    require_benchmark_setup()
    pattern_cfg = missing_rate_to_pattern(SELECTED_SCENARIO, SELECTED_MISSING_RATIO)
    quick_rows = []

    print(f'Quick efficiency scenario: {SELECTED_SCENARIO}')
    print(f'Quick efficiency missing ratio: {SELECTED_MISSING_RATIO}')
    print(f'Quick efficiency datasets: {ROUND2_DATASETS_TO_RUN}')
    print(f'Quick efficiency models: {ROUND2_MODELS_TO_RUN}')

    for dataset_name in ROUND2_DATASETS_TO_RUN:
        for model_name in ROUND2_MODELS_TO_RUN:
            bundle = load_stg_dataset(dataset_name, model_name=model_name)
            _, _, test_loader, _ = make_missing_scenario_loaders(bundle, pattern_cfg)
            model = build_model(
                model_name,
                num_nodes=len(bundle.ci_list),
                A_hat=bundle.A_hat,
                horizon_override=MISSING_EVAL_HORIZON,
            ).to(DEVICE)
            flops = estimate_flops(model, bundle)
            params = count_parameters(model)
            inference_ms = measure_inference_ms(model, test_loader, bundle)
            quick_rows.append({
                'dataset': dataset_name,
                'model': model_name,
                'scenario': SELECTED_SCENARIO,
                'missing_ratio': SELECTED_MISSING_RATIO,
                'flops': flops,
                'parameters': params,
                'inference_ms_per_prediction_round': inference_ms,
            })

    if quick_rows:
        display(pd.DataFrame(quick_rows))
    else:
        print('No rows to display. Enable at least one Round 2 dataset and model.')
else:
    print('Set RUN_ROUND2_EFFICIENCY_CHECK = True to print FLOPs / parameters / inference time for the current Round 2 selection.')



Quick efficiency scenario: RM
Quick efficiency missing ratio: 0.4
Quick efficiency datasets: ['STGdata_C']
Quick efficiency models: ['STG_MFilmFormer']
STGdata_C: T=45,275, cells=16, train_core=[0:20374], val=[20374:22637], test=[22637:31692], simulation=[31692:45275], use_log1p=False, use_zscore=False, model=STG_MFilmFormer


,dataset,model,scenario,missing_ratio,flops,parameters,inference_ms_per_prediction_round
0,STGdata_C,STG_MFilmFormer,RM,0.4,83496960.0,93207,238.551501


## Round 2 STM Sweep Convenience Cell


In [8]:
from __future__ import annotations

# ============================================================
# Round 2. Train RM / TM / SM Missing-Ratio Sweeps In One Go
# ============================================================

# This is a convenience copy of the Round 2 training flow, but fixed to
# RM / TM / SM sweeps over the selected ratio grid. It reuses the exact same
# training and CSV-writing functions as the single-case Round 2 cell.

RUN_ROUND2_SCENARIO_SWEEP = False
ROUND2_SCENARIO_SWEEP_EPOCHS = ROUND2_EPOCHS_TO_RUN if 'ROUND2_EPOCHS_TO_RUN' in globals() else EPOCHS
SCENARIO_SWEEP_GRID = {
    'RM': [0.4, 0.6, 0.8],
    'TM': [0.2, 0.4, 0.6, 0.8],
    'SM': [0.2, 0.4, 0.6, 0.8],
    'STM': [0.2, 0.4, 0.6, 0.8],
}

SCENARIO_SWEEP_DATASETS = ROUND2_DATASETS_TO_RUN if 'ROUND2_DATASETS_TO_RUN' in globals() else ['STGdata_C']
SCENARIO_SWEEP_MODELS = ROUND2_MODELS_TO_RUN if 'ROUND2_MODELS_TO_RUN' in globals() else ['STG_MFilmFormer']

if RUN_ROUND2_SCENARIO_SWEEP:
    if 'train_missing_scenario_experiment' not in globals():
        raise RuntimeError('Please run the Round 2 function cell before launching the scenario sweep.')
    if not SCENARIO_SWEEP_DATASETS or not SCENARIO_SWEEP_MODELS:
        raise RuntimeError('Scenario sweep needs at least one dataset and one model.')

    print('Round 2 scenario sweep grid:', SCENARIO_SWEEP_GRID)
    print('Round 2 scenario sweep datasets:', SCENARIO_SWEEP_DATASETS)
    print('Round 2 scenario sweep models:', SCENARIO_SWEEP_MODELS)

    scenario_sweep_results = []
    scenario_sweep_csvs = {}
    for scenario_name, ratio_list in SCENARIO_SWEEP_GRID.items():
        for scenario_ratio in ratio_list:
            print(f'\n===== {scenario_name} sweep case: {scenario_ratio:.1f} =====')
            for dataset_name in SCENARIO_SWEEP_DATASETS:
                for model_name in SCENARIO_SWEEP_MODELS:
                    scenario_sweep_results.append(
                        train_missing_scenario_experiment(
                            dataset_name,
                            model_name,
                            scenario_name,
                            scenario_ratio,
                            epochs=ROUND2_SCENARIO_SWEEP_EPOCHS,
                        )
                    )
            scenario_sweep_csvs[f'{scenario_name}_{int(round(scenario_ratio * 100))}pct'] = update_missing_scenario_csv_for_case(
                scenario_name,
                scenario_ratio,
                selected_datasets=SCENARIO_SWEEP_DATASETS,
                selected_models=SCENARIO_SWEEP_MODELS,
            )
    print('Round 2 scenario sweep finished.')
else:
    print('Set RUN_ROUND2_SCENARIO_SWEEP = True to train RM / TM / SM sweeps in one run.')



Set RUN_ROUND2_SCENARIO_SWEEP = True to train RM / TM / SM sweeps in one run.


### Round 2 Recovery: refresh all missingness CSVs without retraining



In [9]:
from __future__ import annotations

# ============================================================
# Round 2 Recovery. Refresh All Missingness CSVs Without Retraining
# ============================================================

# Use this after the scenario-specific checkpoints already exist.
# It reloads the saved checkpoints and refreshes every grouped KPI CSV in
# missing_scenario_csvs/ for the selected scenario/rate grid.

REFRESH_ROUND2_ALL_CASE_CSVS = False
RECOVERY_DATASETS = ['STGdata_C']
RECOVERY_MODELS = ['STG_MFilmFormer', 'SBU-LSTM', 'GCNMdynamic', 'GCRNODE']
RECOVERY_SCENARIO_GRID = {
    'RM': [0.2, 0.4, 0.6, 0.8],
    'TM': [0.2, 0.4, 0.6, 0.8],
    'SM': [0.2, 0.4, 0.6, 0.8],
    'STM': [0.2, 0.4, 0.6, 0.8],
}

if REFRESH_ROUND2_ALL_CASE_CSVS:
    require_benchmark_setup()
    print(f'Recovery datasets: {RECOVERY_DATASETS}')
    print(f'Recovery models: {RECOVERY_MODELS}')
    print(f'Recovery scenario grid: {RECOVERY_SCENARIO_GRID}')

    refreshed_tables = {}
    efficiency_rows = []

    for recovery_scenario, ratio_list in RECOVERY_SCENARIO_GRID.items():
        for recovery_missing_ratio in ratio_list:
            print(f'\nRefreshing {recovery_scenario} at {recovery_missing_ratio:.1f}')
            recovery_df = update_missing_scenario_csv_for_case(
                recovery_scenario,
                recovery_missing_ratio,
                selected_datasets=RECOVERY_DATASETS,
                selected_models=RECOVERY_MODELS,
            )
            refreshed_tables[f'{recovery_scenario}_{int(round(recovery_missing_ratio * 100))}pct'] = recovery_df

            pattern_cfg = missing_rate_to_pattern(recovery_scenario, recovery_missing_ratio)
            for dataset_name in RECOVERY_DATASETS:
                for model_name in RECOVERY_MODELS:
                    checkpoint_path = scenario_checkpoint_path(
                        dataset_name,
                        model_name,
                        recovery_scenario,
                        recovery_missing_ratio,
                    )
                    if not checkpoint_path.exists():
                        print(f'Skipping efficiency print, missing checkpoint: {checkpoint_path.name}')
                        continue

                    bundle = load_stg_dataset(dataset_name, model_name=model_name)
                    _, _, test_loader, _ = make_missing_scenario_loaders(bundle, pattern_cfg)
                    payload = torch.load(checkpoint_path, map_location=DEVICE)
                    state_dict = payload['state_dict'] if isinstance(payload, dict) and 'state_dict' in payload else payload
                    model = build_model(
                        model_name,
                        num_nodes=len(bundle.ci_list),
                        A_hat=bundle.A_hat,
                        horizon_override=MISSING_EVAL_HORIZON,
                    ).to(DEVICE)
                    try:
                        model.load_state_dict(state_dict, strict=True)
                    except RuntimeError as exc:
                        print(f'Skipping efficiency print, incompatible checkpoint: {checkpoint_path.name} ({exc})')
                        continue

                    flops = estimate_flops(model, bundle)
                    params = count_parameters(model)
                    inference_ms = measure_inference_ms(model, test_loader, bundle)
                    avg_mae_value = np.nan
                    if 'target_group' in recovery_df.columns and 'average_mae' in recovery_df.columns:
                        match = recovery_df[
                            (recovery_df['dataset'] == dataset_name)
                            & (recovery_df['model'] == model_name)
                            & (recovery_df['scenario'] == recovery_scenario)
                            & (recovery_df['missing_ratio'] == recovery_missing_ratio)
                            & (recovery_df['target_group'] == NEW_ERROR_BALANCED_GROUP)
                        ]
                        if not match.empty:
                            avg_mae_value = match.iloc[0].get('average_mae', np.nan)

                    efficiency_rows.append({
                        'dataset': dataset_name,
                        'model': model_name,
                        'scenario': recovery_scenario,
                        'missing_ratio': recovery_missing_ratio,
                        'flops': flops,
                        'parameters': params,
                        'inference_ms_per_prediction_round': inference_ms,
                        'avg_mae_from_grouped_kpi': avg_mae_value,
                    })

    if refreshed_tables:
        for case_name, case_df in refreshed_tables.items():
            print(f'\nUpdated table: {case_name}')
            display(case_df)
    else:
        print('No grouped KPI tables were refreshed.')

    if efficiency_rows:
        display(pd.DataFrame(efficiency_rows))
    else:
        print('No checkpoint-based efficiency rows to display for the selected recovery grid.')
else:
    print('Set REFRESH_ROUND2_ALL_CASE_CSVS = True to refresh all scenario CSVs from saved checkpoints.')





Set REFRESH_ROUND2_ALL_CASE_CSVS = True to refresh all scenario CSVs from saved checkpoints.


## ploting training losses


In [ ]:
from __future__ import annotations

# ============================================================
# Round 1 Utility. Rebuild Plots From Saved Metrics Without Retraining
# ============================================================

# Run this cell anytime after one or more training runs.
# It reads forecasting_results_missingData/runs/*.csv and recreates dataset-level training-loss PNGs.
# Final test MAE/RMSE/MAPE by horizon stay in forecasting_results_missingData/training_kpi_summary.csv.
plot_paths = plot_all_training_loss_curves()
plot_paths



## inferencing all


In [13]:
from __future__ import annotations

# ============================================================
# Round 1 Utility. Recompute Test KPI With New Errors And Separated Target Groups
# ============================================================

TARGET_GROUPS = {
    "traffic_only": [
        "critical_DL_sum",
        "critical_UL_sum",
        "perform_DL_sum",
        "perform_UL_sum",
    ],
    "emergency_only": [
        "CES_count",
    ],
}
BALANCED_GROUP_NAME = "all_targets_balanced"


def masked_mape_group_np(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> float:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    mask = np.abs(yt) > eps
    if not np.any(mask):
        return float("nan")
    return float(np.mean(np.abs((yp[mask] - yt[mask]) / yt[mask])) * 100.0)


def wape_group_np(y_true: np.ndarray, y_pred: np.ndarray, eps: float = EPS) -> float:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    denom = np.sum(np.abs(yt))
    if denom <= eps:
        return float("nan")
    return float(np.sum(np.abs(yp - yt)) / denom * 100.0)


def metric_four_np(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    return {
        "mae": float(mean_absolute_error(yt, yp)),
        "rmse": float(mean_squared_error(yt, yp, squared=False)),
        "mape": masked_mape_group_np(yt, yp),
        "wape": wape_group_np(yt, yp),
    }


def empty_new_error_row(dataset_name: str, model_name: str, target_group: str) -> Dict[str, object]:
    row = {"dataset": dataset_name, "model": model_name, "target_group": target_group}
    for h in range(1, HORIZON + 1):
        for metric_name in ["mae", "rmse", "mape", "wape"]:
            row[f"horizon_{h}_{metric_name}"] = np.nan
    for metric_name in ["mae", "rmse", "mape", "wape"]:
        row[f"average_{metric_name}"] = np.nan
    return row


@torch.no_grad()
def evaluate_model_grouped(model: nn.Module, loaders: List[DataLoader], bundle: STGDataBundle) -> List[Dict[str, object]]:
    model.eval()
    true_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]
    pred_by_horizon: List[List[np.ndarray]] = [[] for _ in range(HORIZON)]

    for loader in loaders:
        for batch in loader:
            pred, y = forward_model_batch(model, batch, bundle)
            y_true = inverse_target_transform(y, bundle).detach().cpu().numpy()
            y_pred = torch.nan_to_num(inverse_target_transform(pred, bundle), nan=0.0, posinf=1e6, neginf=-1e6).detach().cpu().numpy()
            for h in range(HORIZON):
                true_by_horizon[h].append(y_true[:, h])
                pred_by_horizon[h].append(y_pred[:, h])

    per_group_per_horizon: Dict[str, Dict[str, List[float]]] = {}
    for group_name in list(TARGET_GROUPS.keys()) + [BALANCED_GROUP_NAME]:
        per_group_per_horizon[group_name] = {m: [] for m in ["mae", "rmse", "mape", "wape"]}

    for h in range(HORIZON):
        yt_all = np.concatenate(true_by_horizon[h], axis=0)
        yp_all = np.concatenate(pred_by_horizon[h], axis=0)

        group_metrics: Dict[str, Dict[str, float]] = {}
        for group_name, group_cols in TARGET_GROUPS.items():
            idxs = [TARGET_COLS.index(col) for col in group_cols]
            group_metrics[group_name] = metric_four_np(yt_all[..., idxs], yp_all[..., idxs])
            for metric_name, metric_value in group_metrics[group_name].items():
                per_group_per_horizon[group_name][metric_name].append(metric_value)

        balanced_metrics = metric_four_np(yt_all, yp_all)
        for metric_name, metric_value in balanced_metrics.items():
            per_group_per_horizon[BALANCED_GROUP_NAME][metric_name].append(metric_value)

    rows: List[Dict[str, object]] = []
    for group_name, metric_dict in per_group_per_horizon.items():
        row = {"target_group": group_name}
        for h in range(1, HORIZON + 1):
            for metric_name in ["mae", "rmse", "mape", "wape"]:
                row[f"horizon_{h}_{metric_name}"] = metric_dict[metric_name][h - 1]
        for metric_name in ["mae", "rmse", "mape", "wape"]:
            row[f"average_{metric_name}"] = float(np.nanmean(metric_dict[metric_name]))
        rows.append(row)
    return rows


def run_new_error_kpi_from_checkpoints() -> pd.DataFrame:
    require_benchmark_setup()
    existing_df = pd.read_csv(NEW_ERROR_KPI_CSV) if NEW_ERROR_KPI_CSV.exists() else pd.DataFrame()
    rows: List[Dict[str, object]] = []

    for dataset_name in DATASETS:
        for model_name in MODEL_ORDER:
            bundle = load_stg_dataset(dataset_name, model_name=model_name)
            _, _, test_loader, simulation_loader = make_loaders(bundle)
            checkpoint_path = MODELS_DIR / f"{dataset_name}_{model_name}_best.pt"
            if not checkpoint_path.exists():
                print(f"Missing checkpoint, keeping existing grouped KPI rows if any: {checkpoint_path}")
                continue

            try:
                payload = torch.load(checkpoint_path, map_location=DEVICE)
                state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
                model = build_model(model_name, num_nodes=len(bundle.ci_list), A_hat=bundle.A_hat).to(DEVICE)
                model.load_state_dict(state_dict, strict=True)
                grouped_rows = evaluate_model_grouped(model, [test_loader, simulation_loader], bundle)
                for row in grouped_rows:
                    row["dataset"] = dataset_name
                    row["model"] = model_name
                    rows.append(row)
                print(
                    f"Updated grouped KPI from checkpoint: {checkpoint_path.name} "
                    f"(use_log1p={bundle.use_log1p}, use_zscore={bundle.use_zscore})"
                )
            except Exception as exc:
                print(f"Checkpoint load/eval failed for {checkpoint_path.name}; keeping existing rows: {exc}")
                continue

    new_df = pd.DataFrame(rows)
    preferred = ["traffic_only", "emergency_only", BALANCED_GROUP_NAME]
    key_cols = ["dataset", "model", "target_group"]
    if existing_df.empty:
        df = new_df.copy()
    elif new_df.empty:
        df = existing_df.copy()
    else:
        for col in existing_df.columns:
            if col not in new_df.columns:
                new_df[col] = np.nan
        for col in new_df.columns:
            if col not in existing_df.columns:
                existing_df[col] = np.nan
        existing_df = existing_df[new_df.columns]
        replace_keys = set(tuple(row) for row in new_df[key_cols].itertuples(index=False, name=None))
        keep_mask = [tuple(row) not in replace_keys for row in existing_df[key_cols].itertuples(index=False, name=None)]
        df = pd.concat([existing_df.loc[keep_mask], new_df], ignore_index=True)

    if not df.empty:
        df["target_group"] = pd.Categorical(df["target_group"], categories=preferred, ordered=True)
        df = df.sort_values(["dataset", "model", "target_group"], kind="stable").reset_index(drop=True)
        ordered_cols = ["dataset", "model", "target_group"] + [
            c for c in df.columns if c not in {"dataset", "model", "target_group"}
        ]
        df = df[ordered_cols]
    df.to_csv(NEW_ERROR_KPI_CSV, index=False)
    print(f"Saved grouped test KPI without erasing existing rows: {NEW_ERROR_KPI_CSV}")
    return df


new_error_kpi = run_new_error_kpi_from_checkpoints()
display(new_error_kpi)







STGdata_A: T=50,303, cells=16, train_core=[0:22636], val=[22636:25151], test=[25151:35212], simulation=[35212:50303], use_log1p=True, use_zscore=True, model=DSTAGNN
Windows: train=22,617, val=2,512, test=10,058, simulation=15,088
Missing checkpoint, keeping existing grouped KPI rows if any: forecasting_results_missingData/checkpoints/STGdata_A_DSTAGNN_best.pt
STGdata_A: T=50,303, cells=16, train_core=[0:22636], val=[22636:25151], test=[25151:35212], simulation=[35212:50303], use_log1p=True, use_zscore=True, model=STGFormer
Windows: train=22,617, val=2,512, test=10,058, simulation=15,088
Missing checkpoint, keeping existing grouped KPI rows if any: forecasting_results_missingData/checkpoints/STGdata_A_STGFormer_best.pt
STGdata_A: T=50,303, cells=16, train_core=[0:22636], val=[22636:25151], test=[25151:35212], simulation=[35212:50303], use_log1p=False, use_zscore=False, model=STG_MFilmFormer
STG_MFilmFormer windows: train=22,617, val=2,512, test=10,058, simulation=15,088
Missing checkpoi

/tmp/ipykernel_1205787/562323135.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=DEVICE)


Updated grouped KPI from checkpoint: STGdata_C_DSTAGNN_best.pt (use_log1p=True, use_zscore=True)
STGdata_C: T=45,275, cells=16, train_core=[0:20374], val=[20374:22637], test=[22637:31692], simulation=[31692:45275], use_log1p=True, use_zscore=True, model=STGFormer
Windows: train=20,355, val=2,260, test=9,052, simulation=13,580


/tmp/ipykernel_1205787/562323135.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=DEVICE)


Updated grouped KPI from checkpoint: STGdata_C_STGFormer_best.pt (use_log1p=True, use_zscore=True)
STGdata_C: T=45,275, cells=16, train_core=[0:20374], val=[20374:22637], test=[22637:31692], simulation=[31692:45275], use_log1p=False, use_zscore=False, model=STG_MFilmFormer
STG_MFilmFormer windows: train=20,355, val=2,260, test=9,052, simulation=13,580


...

/tmp/ipykernel_1205787/562323135.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=DEVICE)


Updated grouped KPI from checkpoint: STGdata_C_SBU-LSTM_best.pt (use_log1p=False, use_zscore=False)
STGdata_C: T=45,275, cells=16, train_core=[0:20374], val=[20374:22637], test=[22637:31692], simulation=[31692:45275], use_log1p=False, use_zscore=False, model=GCRNODE
GCRNODE windows: train=20,355, val=2,260, test=9,052, simulation=13,580


/tmp/ipykernel_1205787/562323135.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=DEVICE)


Updated grouped KPI from checkpoint: STGdata_C_GCRNODE_best.pt (use_log1p=False, use_zscore=False)
Saved grouped test KPI without erasing existing rows: forecasting_results_missingData/test_KPI_new_error.csv


,dataset,model,target_group,horizon_1_mae,horizon_1_rmse,horizon_1_mape,horizon_1_wape,horizon_2_mae,horizon_2_rmse,horizon_2_mape,...,horizon_4_wape,average_mae,average_rmse,average_mape,average_wape,horizon_1_maap,horizon_2_maap,horizon_3_maap,horizon_4_maap,average_maap
0,STGdata_C,DSTAGNN,traffic_only,1.104409,18.667881,20.700690,6.331731,2.462889,26.084227,32.867607,...,19.714919,2.440771,27.491466,39.892050,13.994136,NaN,NaN,NaN,NaN,NaN
1,STGdata_C,DSTAGNN,emergency_only,0.003925,0.039700,14.249560,48.621660,0.003485,0.053201,32.624236,...,68.728983,0.004376,0.055514,35.782232,54.213710,NaN,NaN,NaN,NaN,NaN
2,STGdata_C,DSTAGNN,all_targets_balanced,0.884312,16.697067,20.665933,6.336623,1.971009,23.330454,32.866302,...,19.720590,1.953492,24.589127,39.869909,13.998788,NaN,NaN,NaN,NaN,NaN
3,STGdata_C,GCNMdynamic,traffic_only,2.087099,18.990530,56.934899,11.965632,2.361748,24.816364,73.652893,...,20.688021,2.756209,26.819137,98.943734,15.802624,NaN,NaN,NaN,NaN,NaN
4,STGdata_C,GCNMdynamic,emergency_only,0.006266,0.040270,23.738676,77.616113,0.007125,0.051745,35.932907,...,126.415944,0.008170,0.054683,39.621545,101.206893,NaN,NaN,NaN,NaN,NaN
5,STGdata_C,GCNMdynamic,all_targets_balanced,1.670933,16.985659,56.756049,11.973236,1.890823,22.196442,73.449671,...,20.700260,2.206602,23.987776,98.624115,15.812506,NaN,NaN,NaN,NaN,NaN
6,STGdata_C,GCRNODE,traffic_only,3.043460,23.716419,156.605661,17.448585,3.455880,28.818541,190.195608,...,24.485260,3.656522,30.603827,196.067753,20.964470,NaN,NaN,NaN,NaN,NaN
7,STGdata_C,GCRNODE,emergency_only,0.009963,0.046615,32.933033,123.416865,0.012505,0.056837,42.404425,...,212.510800,0.013633,0.059836,47.406611,168.877950,NaN,NaN,NaN,NaN,NaN
8,STGdata_C,GCRNODE,all_targets_balanced,2.436760,21.212618,155.939376,17.460848,2.767206,25.776102,189.399338,...,24.507000,2.927944,27.372908,195.266786,20.981581,NaN,NaN,NaN,NaN,NaN
9,STGdata_C,SBU-LSTM,traffic_only,8.746902,33.415543,363.251185,50.147229,8.966325,36.764553,375.811481,...,57.062852,9.278412,38.213161,380.473155,53.197129,NaN,NaN,NaN,NaN,NaN


### Repair Cell: fill missing efficiency and grouped KPI rows from saved checkpoints



In [ ]:
from __future__ import annotations

# ============================================================
# Repair. Recompute Missing Efficiency / Grouped KPI From Checkpoints
# ============================================================

REPAIR_FROM_CHECKPOINTS = True
REPAIR_DATASETS = ['STGdata_C']
REPAIR_MODELS = ['STGFormer', 'DSTAGNN', 'GCNMdynamic', 'SBU-LSTM', 'GCRNODE', 'STG_MFilmFormer']


if REPAIR_FROM_CHECKPOINTS:
    repaired = []
    for dataset_name in REPAIR_DATASETS:
        for model_name in REPAIR_MODELS:
            checkpoint_path = MODELS_DIR / f'{dataset_name}_{model_name}_best.pt'
            if not checkpoint_path.exists():
                print(f'Missing checkpoint, skipping: {checkpoint_path}')
                continue

            bundle = load_stg_dataset(dataset_name, model_name=model_name)
            _, _, test_loader, simulation_loader = make_loaders(bundle)
            payload = torch.load(checkpoint_path, map_location=DEVICE)
            state_dict = payload['state_dict'] if isinstance(payload, dict) and 'state_dict' in payload else payload
            model = build_model(model_name, num_nodes=len(bundle.ci_list), A_hat=bundle.A_hat).to(DEVICE)
            try:
                model.load_state_dict(state_dict, strict=True)
            except RuntimeError as exc:
                print(f'Skipping checkpoint with incompatible dimensions: {checkpoint_path.name} -> {exc}')
                continue

            flops = estimate_flops(model, bundle)
            params = count_parameters(model)
            inference_ms = measure_inference_ms(model, test_loader, bundle)
            final_metrics = evaluate_model(model, test_loader, bundle, nn.SmoothL1Loss())
            grouped_rows = evaluate_model_grouped(model, [test_loader, simulation_loader], bundle)

            upsert_efficiency_row(dataset_name, model_name, flops, params, inference_ms, final_metrics.average_mae)
            upsert_new_error_rows(dataset_name, model_name, grouped_rows)

            repaired.append({
                'dataset': dataset_name,
                'model': model_name,
                'flops': flops,
                'parameters': params,
                'inference_ms': inference_ms,
                'avg_mae': final_metrics.average_mae,
            })
            print(f'Repaired CSV rows from checkpoint: {checkpoint_path.name}')

    repaired_df = pd.DataFrame(repaired)
    display(repaired_df)
else:
    print('Set REPAIR_FROM_CHECKPOINTS = True to refresh efficiency and grouped KPI rows from saved checkpoints.')

